# improve

> Replay past sessions, and keep the edits to Ramabana's learned folder that make the agent work better.

In [ ]:
#| default_exp improve

In [ ]:
#| hide
from fastcore.test import test_eq, test_fail

Ramabana loads a learned folder into every session: `AGENTS.md`, a few short rules included in every briefing, and skills it opens when a task calls for them. A round tries to improve that folder from one finished session, in the manner of the propose, probe and compose step of Harness². It replays the session's request `k` times from the commit the session started at, under the current folder. An editor model reads those runs and edits a copy of the folder. The request is replayed once more under the copy. The copy is kept only when that run replied, took a cheaper route than every base run, and a judge model rated its reply at least as good as the best base reply. A kept lesson becomes an entry in the day's learning file, committed on a `pending/` branch of the folder's own git repository, one branch per round.

In [ ]:
#| export
import hashlib, importlib, io, json, os, random, re, shutil, subprocess, sys, tarfile, tempfile, time
from collections import Counter
from contextlib import contextmanager, nullcontext
from datetime import datetime, timezone
from fastcore.docments import frontmatter
from fastcore.script import call_parse
from fastcore.xdg import xdg_config_home
from fastcore.xtras import Path

from rahasya import redact
from rahasya.core import IDENTIFYING
from shalya.tools import LESSON_FIELDS, lesson_text, parse_lesson

from dhrona.core import read_history, assess_turn, tool_args, denial_reason

RAMABANA = 'ramabana'   #: the command a round runs; another install replays with that one
LEARNED_MAX = 2000      #: characters of always-on `AGENTS.md` a candidate may carry
SCRUB_KINDS = tuple(sorted(IDENTIFYING - {'address', 'person'}))   #: what `scrub` masks: `address` matches '3 files changed'
_HIDDEN = shutil.ignore_patterns('.*')   # `.git`, and the `.kosha` index Ramabana writes into every folder it opens

def scrub(text):
    "`text` with emails, phone numbers, keys and other identifying values replaced by numbered labels such as `[EMAIL_1]`."
    return redact(str(text or ''), kinds=SCRUB_KINDS, numbered=True)

def _epoch(at):
    "A history row's `at`, epoch seconds or an ISO date, as epoch seconds; 0 when unreadable."
    try: return float(at)
    except (TypeError, ValueError): pass
    try: return datetime.fromisoformat(str(at)).timestamp()
    except ValueError: return 0.

## Runs

`replay_clone` makes a throwaway clone of the repository at the commit a session started from. The clone shares no object files with the source and has no remote, so a replay can neither change nor push to the original. `run_turn` runs one Ramabana turn with no interface, in a fresh config folder, so no history, extension, skill or project note from the person's own config takes part. It reads the turn back from that folder's history, which records the request, every tool call and the reply, and leaves Ramabana's output in `run.log`. Writes are refused unless `approve` allows them. A round's replays pass `read_outside`, so they can read installed packages and other folders the way an interactive session does, while their writes stay inside their clone.

In [ ]:
#| export
def replay_clone(start, dest):
    "A throwaway clone of `start['repo']` checked out at `start['head']`, in `dest`."
    dest = Path(dest)
    dest.parent.mkdir(parents=True, exist_ok=True)
    for cmd in (['git', 'clone', '-q', '--no-hardlinks', start['repo'], str(dest)],   # a replay's edits never reach the source's objects
                ['git', '-C', str(dest), 'checkout', '-q', '--detach', start['head']],
                ['git', '-C', str(dest), 'remote', 'remove', 'origin']):   # a push from a replay has nowhere to go
        if (p := subprocess.run(cmd, stdin=subprocess.DEVNULL, capture_output=True, text=True)).returncode:
            raise RuntimeError(f"could not clone {start['repo']} at {start['head']}: {p.stderr.strip()}")
    return dest

def _kill_tree(p):
    "Kill process `p` and, off Windows, every process in its session: a `claude` or `git` it started too."
    try: os.killpg(p.pid, 9) if os.name != 'nt' else p.kill()
    except (ProcessLookupError, PermissionError): pass

def run_turn(
    prompt,          # the request
    roots,           # folders the turn may work in
    cfg,             # a fresh config folder for this one turn
    model=None,      # Ramabana's routing default when None
    learned=None,    # a learned folder to copy in as `<cfg>/learned`
    approve='ask',   # `ask` refuses every write in a headless turn; `edits` allows file edits
    exe=RAMABANA,
    timeout=1800,    # seconds before the run is abandoned
    read_outside=False, # reads may name any path, as in an interactive session; writes stay in `roots`
):
    "Run one Ramabana turn with no interface and return its history row: `prompt`, `activity`, `reply`, `usage`."
    cfg = Path(cfg)
    cfg.mkdir(parents=True, exist_ok=True)
    if learned is not None: shutil.copytree(learned, cfg/'learned', ignore=_HIDDEN)
    cmd = [exe, '--root', ','.join(map(str, roots)), '--cfg', str(cfg), '--approve', approve,
           '--no-web', '--tmux', 'off', '--pane', 'off', *(['--read-outside'] if read_outside else []),
           *(['--model', model] if model else []), prompt]
    with subprocess.Popen(cmd, stdin=subprocess.DEVNULL, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, start_new_session=os.name != 'nt') as p:
        try: _, err = p.communicate(timeout=timeout)
        except BaseException as e:   # a beat's own deadline as well as ours: what the run started dies with it
            _kill_tree(p); p.communicate()
            if isinstance(e, subprocess.TimeoutExpired): raise RuntimeError(f'the run in {cfg} was still going after {timeout}s') from None
            raise
    (cfg/'run.log').write_text(f'exit {p.returncode}\n\n{err}')
    hist = cfg/'agent-history.jsonl'
    if not hist.exists() or not (turns := read_history(hist)): raise RuntimeError(f'no turn was recorded; see {cfg/"run.log"}')
    return turns[-1]

In [ ]:
from tempfile import TemporaryDirectory
with TemporaryDirectory() as d:
    d = Path(d)
    hang = d/'hung-ramabana'
    hang.write_text(f'#!/bin/sh\nsleep 30 &\necho $! > {d}/child\nwait\n'); hang.chmod(0o755)
    t = time.time()
    test_fail(lambda: run_turn('x', [d], d/'cfg', exe=str(hang), timeout=1), contains='still going after 1s')
    assert time.time() - t < 10
    child = int((d/'child').read_text())
    time.sleep(0.2)
    test_fail(lambda: os.kill(child, 0), exc=ProcessLookupError)   # the run's own child died with it

## Comparing runs

A route's cost is a tuple of its route findings, its failed calls and its calls, and a smaller tuple is a cheaper route. A replay has nobody to answer approvals, so a refused approval does not count as a failure and asking again does not count as a `denial_retry`; in a real session a person would have answered. `delivered` says whether a run produced an answer at all: at least one model turn, a reply, no error, and a `state` of `complete`. A run that was stopped, abandoned or interrupted has no error, but its reply is cut short. `better` passes a candidate only when it delivered and costs less than every base run. Tying the best base run is not enough.

A cheaper route alone proves little, because an agent that gives up early is cheap too. So a round then asks a judge model, through `judge_reply`, to compare every reply, unlabelled and shuffled, and keeps the candidate only when the judge answers YES: its reply is at least as complete and correct as the best base reply. The judge also sees the shell commands each run asked for and was refused, and, when the round has it, what the person's own session ran. A run that proposed the command the work needed is then not marked down for having been refused.

In [ ]:
#| export
def route_cost(turn):
    "`(route findings, failed calls, calls)` for one turn, where refused approvals and asking again do not count and a smaller tuple is cheaper."
    a = assess_turn(turn)
    return (sum(f.kind != 'denial_retry' for f in a.findings),
            sum(x.get('ok') is False and not denial_reason(x) for x in turn.get('activity') or []), a.calls)

def _usage(turn):
    "A turn's `usage` as a dict, whether the row holds a dict or its JSON text."
    u = turn.get('usage')
    if isinstance(u, str):
        try: u = json.loads(u)
        except ValueError: u = None
    return u if isinstance(u, dict) else {}

def delivered(turn):
    "Whether the run took at least one model turn and ended complete, with a reply and no error."
    return (bool((turn.get('reply') or '').strip()) and not turn.get('error') and _usage(turn).get('turns') != 0
            and turn.get('state', 'complete') == 'complete')   # a stopped, abandoned or interrupted run has no error

def better(cand, bases):
    "Whether `cand` delivered a reply and costs less than every turn in `bases`."
    return delivered(cand) and all(route_cost(cand) < route_cost(b) for b in bases)

In [ ]:
def turn(*acts, reply='done'): return {'prompt': 'sizes of the big files', 'reply': reply, 'activity': [dict(tool=t, ok=ok, args=a) for t, ok, a in acts]}
ls, ls2 = ('ls', True, {'path': ''}), ('ls', True, {'path': 'src'})
refused = ('run_shell', False, {'command': 'stat -f %z *.py'})
shell_grep = ('run_shell', True, {'command': 'grep -rn def .'})
base = turn(ls, refused, ls2)
test_eq(route_cost(base), (0, 1, 3))
test_eq(route_cost(turn(shell_grep)), (1, 0, 1))     # a shell grep is a bypass finding
unasked = dict(tool='run_shell', ok=False, args={'command': 'pytest -q'}, detail='Denied by human operator. Reason given: nothing is listening for approvals')
test_eq(route_cost(turn(ls) | {'activity': [dict(tool='ls', ok=True, args={'path': ''}), unasked]}), (0, 0, 2))   # a replay's unanswered approval is not the agent's failure
test_eq(route_cost(turn(ls) | {'activity': [unasked, unasked]}), (0, 0, 2))   # nor is asking again
assert better(turn(ls), [base, turn(ls, ls2)])
assert not better(turn(ls, ls2), [base, turn(ls, ls2)]) # ties the best base run
assert not better(turn(ls, reply=''), [base])
assert not better(turn(reply='claude/sonnet failed (ClaudeError)') | {'usage': {'turns': 0}}, [base])   # no model turn ran: the reply is the failure
assert delivered(turn(ls) | {'usage': '{"turns": 2}'}) and delivered(turn(ls) | {'usage': None})
assert delivered(turn(ls) | {'state': 'complete'}) and not better(turn(ls) | {'state': 'abandoned'}, [base])
for st in ('cancelled', 'abandoned', 'interrupted'): assert not delivered(turn(ls) | {'state': st}), st   # a run cut off, with no error

## The learned folder

A candidate folder may hold `AGENTS.md`, skills at `skills/<name>/SKILL.md` whose frontmatter gives the skill's `name` and a `description`, and Dhrona's dated learning files. `check_learned` lists anything else as a problem, and a round does not run a candidate that has one. `AGENTS.md` may hold `LEARNED_MAX` characters (2,000). The dated files are outside that limit. Hidden entries, such as `.git` and the `.kosha` index Ramabana writes into any folder it opens, are not part of the folder.

`keep_lesson` keeps what a round learned as an entry in `dhrona-<date>.md`, one file a day. An entry holds the request as a heading and the round's notes, and no diff. `keep_lesson` commits the file on a new `pending/<id>` branch, so a person reviews it before it reaches HEAD. It does not check the branch out and does not touch the folder's branch, index or files, so a person's uncommitted edit stays as it was. A batch, a keep and a compaction run whatever the person has left uncommitted. `committed_copy` gives each replay the folder's HEAD, so a replay never sees uncommitted text.

The dated files have their own limit, `DATED_MAX` (4,000 characters across all of them). A keep that would pass it raises `ValueError`, and the round reports the reason in `problems`. Ramabana loads dated files newest first within its own budget, so the limit only stops unbounded growth. A dated file whose date is over `DATED_DAYS` (30) days old is removed by `expire_dated`.

`keep` commits a candidate on a new `pending/<id>` branch of the folder's own repository, as Dhrona, and leaves the folder's files alone. `keep_lesson` and a compaction use it, and a person accepts the branch with `ramabana-learned accept <id>`. `learned_repo` makes the folder a repository the first time and commits what it already held. In an existing repository it leaves uncommitted changes alone.

Personal details never reach the learned folder or the editor and judge. `scrub` masks emails, phone numbers, keys and other identifying values with rahasya, as numbered labels such as `[EMAIL_1]`. It runs over every evidence file, the judge's prompt, the editor's notes and commit message, and every file of a candidate before the candidate is run, so what is tested is what is kept.

In [ ]:
#| export
def _files(folder):
    "Every file under `folder` outside hidden folders, as relative path to text."
    folder = Path(folder)
    return {p.relative_to(folder).as_posix(): p.read_text(errors='replace') for p in sorted(folder.rglob('*'))
            if p.is_file() and not any(s.startswith('.') for s in p.relative_to(folder).parts)}

DATED = re.compile(r'dhrona-\d{4}-\d{2}-\d{2}\.md')   #: the learning files Dhrona writes into the learned folder, one per day
DATED_MAX = 4000   #: characters all dated learning files together may hold; a keep that would pass it is refused
DATED_DAYS = 30    #: a dated learning file whose date is older than this many days is removed

def check_learned(folder):
    "Problems that rule out a candidate learned folder; empty when it may be tried."
    out = []
    for rel, text in _files(folder).items():
        if DATED.fullmatch(rel): continue
        if rel == 'AGENTS.md':
            if len(text) > LEARNED_MAX: out.append(f'AGENTS.md is over {LEARNED_MAX} characters')
        elif re.fullmatch(r'skills/[a-z0-9_-]+/SKILL\.md', rel):
            name, meta = rel.split('/')[1], frontmatter(text)[0]
            if meta.get('name') != name or not meta.get('description'):
                out.append(f'{rel} needs `name: {name}` and a `description` in its frontmatter')
        else: out.append(f'{rel} is neither AGENTS.md nor skills/<name>/SKILL.md')
    return out

def _uncommitted(learned):
    "Whether the learned folder's own repository has uncommitted changes; an untracked hidden path, such as Ramabana's `.kosha` index, is not one."
    learned = Path(learned)
    if not (learned/'.git').exists(): return False
    out = subprocess.run(['git', '-C', str(learned), 'status', '--porcelain'], stdin=subprocess.DEVNULL, capture_output=True, text=True).stdout
    return any(l.strip() and not l.startswith('?? .') for l in out.splitlines())

PENDING = 'pending/'   #: the branch prefix a kept candidate waits under until a person accepts it into HEAD
LESSON = 'Lesson from a replay'   #: how the message of a commit Dhrona proposed begins; only those lines and skills expire
COMPACT = 'Compact learned AGENTS.md'   #: how the message of a compaction's commit begins; what it wrote counts as Dhrona's own

def _is_dhronas(subj): return subj.startswith((LESSON, COMPACT))

_AS_DHRONA = ('-c', 'user.name=Dhrona', '-c', 'user.email=dhrona@localhost')

def _git(d, *a): return subprocess.run(['git', '-C', str(d), *a], check=True, stdin=subprocess.DEVNULL, capture_output=True, text=True).stdout.strip()

def _show(learned, rev, rel='AGENTS.md'): return subprocess.run(['git', '-C', str(learned), 'show', f'{rev}:{rel}'], stdin=subprocess.DEVNULL, capture_output=True, text=True).stdout

def learned_repo(learned):
    "`learned` as its own git repository with a first commit holding what it already had; a person's uncommitted changes in an existing repository stay as they are."
    learned = Path(learned)
    learned.mkdir(parents=True, exist_ok=True)
    if not (learned/'.git').exists():
        _git(learned, 'init', '-q')
        with open(learned/'.git'/'info'/'exclude', 'a') as f: f.write('.kosha/\n')   # Ramabana's code index is not a lesson
        _git(learned, 'add', '-A'); _git(learned, *_AS_DHRONA, 'commit', '-q', '--allow-empty', '-m', 'The learned folder before Dhrona kept a round')
    _git(learned, 'worktree', 'prune')   # a worktree a crashed round left
    return learned

def keep(candidate, learned, message, scrubbed=True):
    "Commit `candidate`'s files on a new `pending/<id>` branch off `learned`'s HEAD, leaving its working tree alone; returns the branch."
    learned = learned_repo(learned)
    branch = PENDING + datetime.now().strftime('%Y%m%d-%H%M%S-') + os.urandom(3).hex()
    wt = Path(tempfile.mkdtemp())/'wt'
    _git(learned, 'worktree', 'add', '-q', '-b', branch, str(wt), 'HEAD')
    try:
        for p in wt.iterdir():
            if not p.name.startswith('.'): shutil.rmtree(p) if p.is_dir() else p.unlink()
        shutil.copytree(candidate, wt, dirs_exist_ok=True, ignore=_HIDDEN)
        _git(wt, 'add', '-A'); _git(wt, *_AS_DHRONA, 'commit', '-q', '--allow-empty', '-m', scrub(message) if scrubbed else message)   # `scrub` reads some pending ids as card numbers
    finally: subprocess.run(['git', '-C', str(learned), 'worktree', 'remove', '--force', str(wt)], stdin=subprocess.DEVNULL, capture_output=True)
    return branch

def committed_copy(learned, dest):
    "`learned`'s HEAD as the new folder `dest`, so a person's uncommitted edits are never in it; a folder that is no repository is copied whole."
    learned, dest = Path(learned), Path(dest)
    if not (learned/'.git').exists(): return shutil.copytree(learned, dest, ignore=_HIDDEN)
    dest.mkdir(parents=True)
    head = subprocess.run(['git', '-C', str(learned), 'ls-tree', '-r', '--name-only', 'HEAD'], stdin=subprocess.DEVNULL, capture_output=True, text=True)
    if head.stdout.strip():   # no HEAD, or one with an empty tree, leaves `dest` empty: `tarfile` refuses that archive
        tarfile.open(fileobj=io.BytesIO(subprocess.run(['git', '-C', str(learned), 'archive', 'HEAD'], stdin=subprocess.DEVNULL, capture_output=True, check=True).stdout)).extractall(dest, filter='data')
    return dest

def lesson_entry(prompt, notes):
    "One kept lesson as markdown for a dated file: the request in a heading, then the round's notes."
    return f"## {' '.join(prompt.split())[:60]}\n\n{notes.strip()}"

def keep_lesson(learned, entry, message):
    "Add `entry` to today's `dhrona-<date>.md` on a new `pending/<id>` branch off `learned`'s HEAD and return the branch; raises `ValueError` when the dated files would pass `DATED_MAX`. The folder's branch, index and files are not touched."
    learned = learned_repo(learned)
    name = f'dhrona-{datetime.now():%Y-%m-%d}.md'
    dated = {n: _show(learned, 'HEAD', n) for n in _git(learned, 'ls-tree', '--name-only', 'HEAD').split('\n') if DATED.fullmatch(n)}
    refs = _git(learned, 'for-each-ref', '--format=%(refname)', f'refs/heads/{PENDING}{datetime.now():%Y%m%d}-*').split()
    base = max([dated.get(name, ''), *(_show(learned, r, name) for r in refs)], key=len) or f'# Dhrona lessons {name[7:-3]}\n'
    text = base + '\n' + scrub(entry).strip() + '\n'
    if (n := sum(len(t) for f, t in dated.items() if f != name) + len(text)) > DATED_MAX:
        raise ValueError(f'the lesson would take the dated learning files to {n:,} characters, past the {DATED_MAX:,} allowed')
    with tempfile.TemporaryDirectory() as t:
        cand = committed_copy(learned, Path(t)/'cand')
        (cand/name).write_text(text)
        return keep(cand, learned, message)

def _scrub_folder(folder):
    "Rewrite each file under `folder` through `scrub`; returns the ones it changed."
    out = []
    for rel, text in _files(folder).items():
        if (s := scrub(text)) != text: (Path(folder)/rel).write_text(s); out.append(rel)
    return out

In [ ]:
from tempfile import TemporaryDirectory
with TemporaryDirectory() as d:
    d = Path(d); cand, learned = d/'cand', d/'learned'
    (cand/'skills'/'sizes').mkdir(parents=True)
    (cand/'AGENTS.md').write_text('Read file sizes from `ls`; it reports them.\n')
    (cand/'skills'/'sizes'/'SKILL.md').write_text('---\nname: sizes\ndescription: Compare file sizes.\n---\nUse ls.\n')
    (cand/'.kosha').mkdir(); (cand/'.kosha'/'code.db').write_bytes(b'\x95\x00')   # the index Ramabana writes into a folder it opens
    test_eq(check_learned(cand), [])
    (cand/'notes.txt').write_text('x')
    (cand/'skills'/'bad').mkdir(); (cand/'skills'/'bad'/'SKILL.md').write_text('no frontmatter')
    test_eq(len(check_learned(cand)), 2)
    (cand/'notes.txt').unlink(); shutil.rmtree(cand/'skills'/'bad')
    (learned/'skills'/'stale').mkdir(parents=True); (learned/'skills'/'stale'/'SKILL.md').write_text('old')
    b = keep(cand, learned, 'Read sizes from ls; ask ann@example.com')
    assert b.startswith(PENDING)
    test_eq(sorted(_files(learned)), ['skills/stale/SKILL.md'])   # the working tree, which Ramabana reads, is untouched
    git = lambda *a: _git(learned, *a)
    assert 'skills/stale/SKILL.md' in git('show', '--stat', 'HEAD')   # what was there before Dhrona is the first commit
    test_eq(git('ls-tree', '-r', '--name-only', b).split(), ['AGENTS.md', 'skills/sizes/SKILL.md'])
    test_eq(git('log', '-1', '--format=%an %s', b), 'Dhrona Read sizes from ls; ask [EMAIL_1]')
    test_eq(len(git('worktree', 'list').splitlines()), 1)   # the worktree it committed in is gone
    (learned/'AGENTS.md').write_text('a hand edit')
    assert keep(cand, learned, 'again') != b   # a hand edit does not stop a keep, and the keep leaves it alone
    test_eq((learned/'AGENTS.md').read_text(), 'a hand edit')
    (learned/'AGENTS.md').unlink()
    assert not _uncommitted(learned_repo(d/'empty')) and _git(d/'empty', 'rev-parse', 'HEAD')   # an empty folder gets an empty first commit
    (cand/'AGENTS.md').write_text('Mail ann@example.com or call +1 415 555 0134.\n')
    test_eq(_scrub_folder(cand), ['AGENTS.md'])
    test_eq((cand/'AGENTS.md').read_text(), 'Mail [EMAIL_1] or call [PHONE_1].\n')

Ramabana writes a `.kosha` code index into every folder it opens, so one can sit untracked in the learned folder. Untracked hidden paths are not a person's edit, as `_files` and `keep` already treat them, and `learned_repo` keeps `.kosha/` out of a new repository's status. A modified `AGENTS.md` does not block a batch, a keep or a compaction: Dhrona never stages, commits or discards a person's edit.

In [ ]:
with TemporaryDirectory() as d:
    d = Path(d); learned = d/'learned'
    learned.mkdir(); (learned/'AGENTS.md').write_text('Read sizes from `ls`.\n')
    learned_repo(learned)
    (learned/'.kosha').mkdir(); (learned/'.kosha'/'code.db').write_bytes(b'\x95\x00')
    assert '.kosha/' in (learned/'.git'/'info'/'exclude').read_text()
    (learned/'.git'/'info'/'exclude').write_text('')   # a repository made before the exclude line existed
    test_eq(_git(learned, 'status', '--porcelain'), '?? .kosha/')
    test_eq(_uncommitted(learned), False)
    test_eq(learned_repo(learned), learned)
    (learned/'AGENTS.md').write_text('a hand edit')
    test_eq(_uncommitted(learned), True)
    test_eq(learned_repo(learned), learned)   # an edit is not a reason to refuse
    status = _git(learned, 'status', '--porcelain')
status

A kept lesson is a short entry in `dhrona-<date>.md`: the request as a heading and the round's notes under it, with no diff of the files the editor changed. `keep_lesson` commits that file on a new `pending/<id>` branch off HEAD, in a temporary worktree. It does not check the branch out, and it does not touch the learned folder's checked-out branch, index or files, so a person's uncommitted edit stays as it was. A second lesson on the same day goes on its own branch off the same HEAD. Its file holds both entries, because it starts from the first branch's file.

In [ ]:
with TemporaryDirectory() as d:
    d = Path(d); learned = d/'learned'
    learned.mkdir(); (learned/'AGENTS.md').write_text('Read sizes from `ls`.\n')
    learned_repo(learned)
    (learned/'AGENTS.md').write_text('a hand edit\n')   # a person's uncommitted edit
    (learned/'notes.txt').write_text('draft'); _git(learned, 'add', 'notes.txt')   # and a staged file
    state = lambda: (_git(learned, 'rev-parse', 'HEAD'), _git(learned, 'branch', '--show-current'), _git(learned, 'status', '--porcelain'),
                     (learned/'AGENTS.md').read_text(), sorted(_files(learned)))
    before = state()
    entry = lesson_entry('sizes of the big files', 'Read file sizes with `ls`. Ask ann@example.com.')
    b = keep_lesson(learned, entry, 'Lesson from a replay: sizes of the big files')
    assert b.startswith(PENDING)
    test_eq(state(), before)   # branch, index, working tree and HEAD are as they were
    name = f'dhrona-{datetime.now():%Y-%m-%d}.md'
    test_eq(_git(learned, 'diff', '--name-only', 'HEAD', b), name)
    text = _git(learned, 'show', f'{b}:{name}')
    assert 'Read file sizes with `ls`' in text and '[EMAIL_1]' in text and not any(m in text for m in ('@@', '+++', '```diff'))
    b2 = keep_lesson(learned, lesson_entry('a second request', 'Use `ls -l`.'), 'Lesson from a replay: a second request')
    test_eq(_git(learned, 'rev-parse', f'{b2}^'), _git(learned, 'rev-parse', 'HEAD'))
    test_eq(len(_git(learned, 'worktree', 'list').splitlines()), 1)   # no worktree is left behind
    test_eq(state(), before)
    both = _git(learned, 'show', f'{b2}:{name}')
test_eq(both.count('\n## '), 2)
print(both)

`LEARNED_MAX` (2,000 characters) counts `AGENTS.md` alone. Dated learning files have their own cap, `DATED_MAX` (4,000 characters across all of them). `keep_lesson` raises `ValueError` for a lesson that would take the dated files past it, and a round reports that reason in `problems`. Ramabana loads dated files newest first within its own budget, so the cap only stops unbounded growth.

In [ ]:
with TemporaryDirectory() as d:
    d = Path(d); learned = d/'learned'
    learned.mkdir(); (learned/'AGENTS.md').write_text('x' * LEARNED_MAX)
    (learned/'dhrona-2026-01-01.md').write_text('y' * (DATED_MAX - 100))
    test_eq(check_learned(learned), [])   # dated files are outside `LEARNED_MAX`
    (learned/'AGENTS.md').write_text('x' * (LEARNED_MAX + 1))
    test_eq(len(check_learned(learned)), 1)
    (learned/'AGENTS.md').write_text('x' * LEARNED_MAX)
    learned_repo(learned)
    test_fail(lambda: keep_lesson(learned, 'z' * 200, 'Lesson from a replay: long'), contains=f'{DATED_MAX:,}')
    test_eq(_git(learned, 'branch', '--list', 'pending/*'), '')   # a refused lesson leaves no branch
    b = keep_lesson(learned, lesson_entry('short', 'z' * 20), 'Lesson from a replay: short')
    files = _git(learned, 'ls-tree', '--name-only', b)
files

## Evidence for the editor

The editor model reads files, not one long prompt. `trajectory_md` writes one turn as markdown: the request, each tool call with its arguments, outcome and shortened result, and the reply. `findings_md` lists each run's route cost, its findings and its failed or refused calls. `MISSION` is the editor's prompt. It says where the evidence and the candidate folder are, what the editor may change, and that changing nothing is a valid answer.

In [ ]:
#| export
def trajectory_md(turn, mx=1500):
    "One turn as markdown: the request, each tool call with its arguments, outcome and clipped result, then the reply."
    rows = [f"## Request\n\n{turn.get('prompt', '')}", '## Tool calls']
    for i, a in enumerate(turn.get('activity') or []):
        out = str(a.get('detail') or '')[:mx].replace('\n', '\n    ')
        rows.append(f"{i}. `{a.get('tool')}` {json.dumps(tool_args(a), default=str)} -> {'ok' if a.get('ok') else 'failed'}\n\n    {out}")
    rows.append(f"## Reply\n\n{turn.get('reply') or '(no reply)'}")
    return '\n\n'.join(rows)

def findings_md(turns):
    "Each run's route cost, findings, and failed or refused calls, as markdown."
    rows = []
    for n, t in enumerate(turns, 1):
        acts = t.get('activity') or []
        rows.append(f'## base-{n}: route cost {route_cost(t)} (findings, failed calls, calls)')
        rows += [f'- finding at call {f.index}: {f.message}' for f in assess_turn(t).findings]
        rows += [f"- call {i} `{a.get('tool')}` {'refused' if denial_reason(a) else 'failed'}: {str(a.get('detail') or '')[:300]}"
                 for i, a in enumerate(acts) if a.get('ok') is False]
    return '\n'.join(rows)

MISSION = r"""You improve how a coding agent, Ramabana, works by editing its learned folder, `{candidate}`.

Ramabana puts `AGENTS.md` from that folder into every briefing, after its own rules and before the project's instructions. It offers each `skills/<name>/SKILL.md` there as an on-demand skill, listed by the `name` and `description` in its frontmatter. Nothing else in the folder is read.

The evidence is in `{evidence}`. `task.md` is a request a person made. Each `base-N.md` is one run of that request under the folder as it stands. `findings.md` gives each run's route cost and what Dhrona's route check flagged. Read every evidence file and the current folder before you decide.

Look for a behaviour that repeats across the runs and costs the route something: a call that was refused or failed, a shell command where a dedicated tool does the job, or a call that repeats work already done. A defect in one run only is noise.

The runs were replays with nobody present, so every `run_shell` call was refused for want of an approver. In a real session a person approves or declines each shell command. A lesson must hold in a real session: write nothing about refused approvals, and never steer the agent away from a shell command the work needs. Redirect a shell command only when a dedicated tool does the same job. A finding is a hint, not a verdict; if no dedicated tool does what the shell command did, the finding is wrong.

`original.md`, when it is there, is the person's own session of this request, in which the person approved or declined each shell command. Its results show what a refused command in a replay would have done, and whether the person wanted it. Use it to judge whether a run's refused command was the right action. Never write a lesson that steers the agent away from a command the person approved.

`lessons.md`, when it is there, lists lessons the agent recorded with `remember` in recent sessions: what to do, the situation it applies to, the evidence and how sure it was. Treat a lesson like a finding: act on it only when the runs show the same behaviour. A lesson under "Tool defects" describes a tool that misbehaved. When the runs hit the same defect, write a skill named for that tool that tells the agent how to get the result despite it.

Values such as `[EMAIL_1]` stand for details removed from the evidence. Never write them, or any personal detail, into the folder.

Change only what the evidence shows needs changing.
- Prefer a skill for a procedure with a recognisable trigger. It costs nothing until it is used.
- Use `AGENTS.md` only for a short rule that applies to every task. Keep the whole file under {mx} characters, and cut something to pay for what you add.
- Write nothing specific to this request or this repository. The folder applies to every project.
- Never change what the agent delivers, only how it works.

If the evidence argues for no change, change nothing. That is a valid answer.

Edit the files in `{candidate}` with your tools. Then reply with one line per change naming the run and tool call that motivated it, or with the single word KEEP if you changed nothing."""

In [ ]:
print(findings_md([base]))
print(trajectory_md(base)[:400])

In [ ]:
#| export
def _history(cfg):
    "Ramabana's turn rows in `cfg`, or none when it has no history yet."
    p = Path(cfg).expanduser()/'agent-history.jsonl'
    return read_history(p) if p.exists() else []

def recent_lessons(cfg='~/.config/ramabana', days=30, now=None):
    "Lessons `remember` recorded in Ramabana's history in the last `days` days, newest first, one per lesson text, scrubbed."
    now = now or time.time()
    cutoff, out, seen = now - days*86400, [], set()
    for t in reversed(_history(cfg)):
        if not cutoff <= _epoch(t.get('at')) <= now: continue
        for a in reversed(t.get('activity') or []):
            if a.get('tool') != 'remember' or a.get('ok') is False: continue
            args = tool_args(a)
            try: d = parse_lesson(lesson_text(*(str(args.get(k) or '') for k in ('text', *LESSON_FIELDS[1:]))))   # the text either memory layout stores
            except ValueError: continue
            if d is None or (k := ' '.join(d['lesson'].lower().split())) in seen: continue
            seen.add(k)
            out.append({f: scrub(v) for f, v in d.items()} | dict(at=t.get('at'), session=t.get('session')))
    return out

def lessons_md(lessons):
    "Lessons as markdown, tool defects first."
    def row(l): return f"- {l['lesson']}" + ''.join(f"\n  {f}: {l[f]}" for f in LESSON_FIELDS[1:] if l.get(f) and f != 'component')
    tool, rest = [l for l in lessons if l.get('component') == 'tool'], [l for l in lessons if l.get('component') != 'tool']
    return '\n\n'.join(f'## {h}\n\n' + '\n'.join(map(row, ls)) for h, ls in (('Tool defects', tool), ('Other lessons', rest)) if ls)

`recent_lessons` reads the lessons the agent recorded with `remember` from Ramabana's history. Every `remember` call is in the history whether the session had a vault or only a memory file, and shalya's `parse_lesson` reads the fields back the same way for both. Only structured lessons count, one per lesson text, newest first, from the last `days` days, with personal details scrubbed. `lessons_md` writes them for the editor as `lessons.md`, tool defects first, because a tool defect is a candidate for a skill that works around it.

In [ ]:
with TemporaryDirectory() as d:
    cfg = Path(d)
    rem = lambda **a: dict(tool='remember', ok=True, args=a)
    rows = [dict(session='s1', at=time.time() - 40*86400, activity=[rem(text='old lesson', cue='c', component='rule')]),
            dict(session='s2', at=time.time() - 60, activity=[rem(text='a plain note'),
                 rem(text='filter on the root-relative path', cue='empty search', component='tool', evidence='mail ann@example.com said so', confidence='high'),
                 rem(text='bad', component='nonsense'), dict(tool='remember', ok=False, args={'text': 'refused', 'cue': 'x'})]),
            dict(session='s3', at=time.time() - 30, activity=[rem(text='Filter on the  root-relative path', cue='again')])]
    (cfg/'agent-history.jsonl').write_text('\n'.join(map(json.dumps, rows)) + '\n')
    got = recent_lessons(cfg)
    test_eq([(l['lesson'], l['session']) for l in got], [('Filter on the root-relative path', 's3')])   # newest wins; old, plain, invalid and refused are left out
    got = recent_lessons(cfg, now=time.time() - 45)
    test_eq(got[0]['evidence'], 'mail [EMAIL_1] said so')
    md = lessons_md(got + [dict(lesson='keep notes short', component='rule')])
    assert md.index('## Tool defects') < md.index('filter on the root-relative path') < md.index('## Other lessons') and 'ann@' not in md

In [ ]:
#| export
JUDGE = r"""A person asked:

{request}

Here are {n} answers to that request from separate runs, labelled in no particular order. The runs had nobody present to approve shell commands, so each shell command a run asked to run was refused. Those commands are listed under its answer: they are what the run would have done with approval.

{answers}
{original}
Does answer {label} answer the request at least as completely and correctly as the best of the other answers? A shorter answer is fine when it is complete. An answer that stops to ask for something another answer found is less complete. A proposed command counts as part of its answer: one the request needed, stated so the person can run it, counts as done; one the request did not need, or one that would do harm, counts against it. Judge only what is written here, and do not use tools.

Reply with YES or NO on the first line, then one sentence giving the reason."""

def refused_commands(turn):
    "The shell commands `turn` asked to run and was refused for want of an approver."
    return [str(tool_args(a).get('command') or '') for a in turn.get('activity') or [] if a.get('tool') == 'run_shell' and denial_reason(a)]

def ran_commands(turn, mx=600):
    "`(command, outcome, output)` for each shell command `turn` asked for: `ran`, `failed`, or `declined: ` and the person's reason."
    return [(str(tool_args(a).get('command') or ''), f'declined: {r}' if (r := denial_reason(a)) else 'ran' if a.get('ok') else 'failed',
             '' if r else str(a.get('detail') or '')[:mx]) for a in turn.get('activity') or [] if a.get('tool') == 'run_shell']

def _commands_md(cmds): return '\n'.join(f'- `{c}`' for c in cmds)

def judge_prompt(request, cand, bases, original=None):
    "`(prompt, label)`: the judge's prompt over every reply in an order fixed by `request`, and the label of the candidate's reply."
    turns = [*bases, cand]
    order = list(range(len(turns)))
    random.Random(request).shuffle(order)
    labels = [chr(ord('A') + j) for j in range(len(order))]
    label = labels[order.index(len(turns) - 1)]
    def answer(j, i):
        asked = refused_commands(turns[i])
        return f"## Answer {labels[j]}\n\n{turns[i].get('reply') or ''}" + (f'\n\nCommands it asked to run:\n{_commands_md(asked)}' if asked else '')
    seen = ran_commands(original) if original else []
    own = ('\n## Commands in the person\'s own session of this request\n\nThe person approved or declined each one there. They show what the work needed.\n\n'
           + '\n'.join(f'- `{c}` ({how})' + (f": {' '.join(out.split())}" if out else '') for c, how, out in seen) + '\n') if seen else ''
    answers = '\n\n'.join(answer(j, i) for j, i in enumerate(order))
    return JUDGE.format(request=request, n=len(turns), answers=answers, original=own, label=label), label

def _says_yes(text):
    "Whether the first line of a judge's reply answers YES."
    return bool(re.match(r'\W*yes\b', (text or '').strip().split('\n')[0], re.I))

def judge_reply(
    request,        # what the person asked
    cand,           # the candidate run's turn
    bases,          # the base runs' turns
    folder,         # a new folder for the judge's run
    model=None,     # the model that judges
    exe=RAMABANA,
    original=None,  # the person's own turn for this request, whose commands show what the work needed
):
    "Ask a judge model whether `cand`'s reply is at least as good as the best of `bases`, and return `(kept, reply)`."
    prompt = scrub(judge_prompt(request, cand, bases, original)[0])   # the judge sees no personal detail
    folder = Path(folder)
    folder.mkdir(parents=True)
    t = run_turn(prompt, [folder], folder/'cfg', model, exe=exe)
    if not delivered(t): raise RuntimeError(f"the judge gave no answer: {t.get('error') or t.get('reply') or ''}")
    return _says_yes(t['reply']), t['reply']

A replay refuses every shell command, because nobody is there to approve it, but the command a run asked for still shows what it meant to do. `refused_commands` lists those commands, and `judge_prompt` puts them under that run's answer, so the judge can count a needed command, written out for the person to run, as done. `ran_commands` lists what the person's own session ran, what failed and what the person declined, with the reason. `judge_prompt` shows these as what the work needed, and the editor reads the same session as `original.md`.

In [ ]:
def tn(*acts, reply='done'): return {'prompt': 'tidy the drafts', 'reply': reply, 'activity': list(acts)}
look = dict(tool='ls', ok=True, args={'path': ''})
asked = dict(tool='run_shell', ok=False, args={'command': 'git add notes.md && rm draft.md'},
             detail='Denied by human operator. Reason given: nothing is listening for approvals')
broke = dict(tool='run_shell', ok=False, args={'command': 'pytest -q'}, detail='ERROR: exit 1')
ran = dict(tool='run_shell', ok=True, args={'command': 'git add notes.md && rm draft.md'}, detail='(no output)')
nope = dict(tool='run_shell', ok=False, args={'command': 'git push'}, detail='Denied by human operator. Reason given: not yet')
test_eq(refused_commands(tn(look, asked, broke)), ['git add notes.md && rm draft.md'])   # one that ran and failed was not refused
test_eq(ran_commands(tn(look, ran, broke, nope)),
        [('git add notes.md && rm draft.md', 'ran', '(no output)'), ('pytest -q', 'failed', 'ERROR: exit 1'), ('git push', 'declined: not yet', '')])
proposer = tn(look, asked, reply='Run `git add notes.md && rm draft.md`; I could not.')
avoider = tn(look, reply='Done.')
p, label = judge_prompt('tidy the drafts', proposer, [avoider], original=tn(ran))
other = 'B' if label == 'A' else 'A'
blocks = {l: p.split(f'## Answer {l}\n')[1].split('\n## ')[0] for l in 'AB'}
assert 'rm draft.md' in blocks[label] and 'rm draft.md' not in blocks[other], blocks   # under the run that asked for it
assert "person's own session" in p and '(no output)' in p
bare, _ = judge_prompt('tidy the drafts', avoider, [avoider])
assert 'Commands' not in bare and "person's own session" not in bare   # nothing refused and no session: neither section

## A round

`improve_round` takes a request and the `start` record Ramabana wrote for its session, `<cfg>/runs/<session>/start.json`, which holds `repo`, `head` and `dirty`. It raises `ValueError` before any run starts when:

- the start had uncommitted changes, which a replay from `head` would not see,
- the learned folder holds files outside the allowed layout,
- `approve` is anything but `ask` or `edits`, so a replay never runs a shell command,
- `k` is less than 1.

`work` must be a new folder, or the round raises `FileExistsError`. Every run, the evidence, the candidate and `report.json` stay in it. If a base run's model never answered, the round ends with `base_error` before the editor runs. If the editor or the candidate run records no turn or gives no answer, the round ends with that error in the report. By default replays refuse every write. `approve='edits'` lets them edit files inside their own clone, which shares no object files with the source repository. With `learned` set, a kept lesson goes into the day's `dhrona-<date>.md` on a `pending/` branch there, and the report names the branch as `pending`; without it, the round only reports. A lesson that would take the dated files past `DATED_MAX` characters is not kept, and the report lists that in `problems`. `lessons`, rows from `recent_lessons`, go to the editor as `lessons.md`.

The tests below use a stand-in script for `ramabana` that answers each kind of prompt with a fixed reply. Under it, a run takes one call once `AGENTS.md` exists and three calls before.

In [ ]:
#| export
def _summary(turn):
    "What a report keeps of one turn."
    return dict(route=route_cost(turn), findings=[f.message for f in assess_turn(turn).findings],
                calls=[a.get('tool') for a in turn.get('activity') or []], reply=turn.get('reply'),
                cost=_usage(turn).get('cost'))

def round_cost(work):
    "Dollars the runs in round folder `work` recorded, whether or not the round finished; a run that recorded no turn adds nothing."
    return sum(c for h in Path(work).glob('*/cfg/agent-history.jsonl') if (ts := read_history(h)) and (c := _usage(ts[-1]).get('cost')))

ROUND_LIMIT = 8   #: dollars: a round stops once the runs it has recorded cost more than this

def improve_round(
    prompt,         # the request to replay
    start,          # `{repo, head, dirty}` from the session's `start.json`
    work,           # a new folder for this round's runs, evidence and report
    learned=None,   # the learned folder whose HEAD the round starts from, and whose `pending/` branch a kept lesson goes onto
    model=None,     # the model the runs use
    editor=None,    # the model that edits the candidate
    k=2,            # base runs
    approve='ask',  # `edits` lets a replay edit files inside its throwaway clone
    exe=RAMABANA,
    read_outside=True, # replays read outside their clone, as a session started with `--read-outside` does
    original=None,  # the person's own turn for this request; the editor and the judge see what its commands did
    lessons=(),     # `recent_lessons` rows the editor reads as evidence
    timeout=1800,   # seconds before a replay, base or candidate, is abandoned
    limit=ROUND_LIMIT, # dollars the round's recorded runs may cost before it stops
):
    "Run one round of `k` base runs, one edit to the learned folder, one run under the edit and the decision, and return the report."
    if k < 1: raise ValueError('a round needs at least one base run')
    if approve not in ('ask', 'edits'): raise ValueError(f"approve is 'ask' or 'edits', not {approve!r}: a replay never runs shell commands")
    if start.get('dirty'): raise ValueError(f"{start['repo']} had uncommitted changes when the session began; a replay from {start['head']} would not see them")
    work = Path(work)
    if work.exists(): raise FileExistsError(f'{work} exists; a round writes into a new folder')
    current, cand, ev = work/'current', work/'candidate', work/'evidence'
    if learned is not None and Path(learned).exists(): committed_copy(learned, current)
    else: current.mkdir(parents=True)
    if (bad := check_learned(current)): raise ValueError(f'{learned} holds files a learned folder may not: {bad}')
    def run(name, folder): return run_turn(prompt, [replay_clone(start, work/name/'repo')], work/name/'cfg', model, folder, approve, exe=exe,
                                           timeout=timeout, read_outside=read_outside)
    bases = []
    for n in range(1, k+1):
        bases.append(run(f'base-{n}', current))
        if round_cost(work) > limit: break
    report = dict(prompt=prompt, start=start, bases=[_summary(b) for b in bases], adopted=False)
    def capped():
        "Whether the runs so far cost more than `limit`; the report then says so and the round ends."
        if (spent := round_cost(work)) <= limit: return False
        report['cost_limit'] = f'the round cost ${spent:.2f}, past its ${limit:g} limit'
        (work/'report.json').write_text(json.dumps(report, indent=1, default=str))
        return True
    if capped(): return report
    if (dead := [n for n, b in enumerate(bases, 1) if not delivered(b)]):
        first = bases[dead[0] - 1]
        report['base_error'] = f"base runs {dead} gave no answer: {(first.get('error') or first.get('reply') or '')[:300]}"
        (work/'report.json').write_text(json.dumps(report, indent=1, default=str))
        return report
    ev.mkdir()
    (ev/'task.md').write_text(scrub(prompt))   # the editor reads no personal detail
    for n, b in enumerate(bases, 1): (ev/f'base-{n}.md').write_text(scrub(trajectory_md(b)))
    (ev/'findings.md').write_text(scrub(findings_md(bases)))
    if original: (ev/'original.md').write_text(scrub("# The person's own session of this request\n\nA person approved or declined each shell command here.\n\n" + trajectory_md(original)))
    if lessons: (ev/'lessons.md').write_text(scrub(lessons_md(lessons)))
    shutil.copytree(current, cand, ignore=_HIDDEN)
    try: edit = run_turn(MISSION.format(candidate=cand, evidence=ev, mx=LEARNED_MAX), [cand, ev], work/'editor'/'cfg', editor,
                         approve='edits', exe=exe)
    except RuntimeError as e: edit = dict(error=str(e))
    editor_error = edit.get('error') or (None if delivered(edit) else f"the editor gave no answer: {edit.get('reply') or ''}")
    report |= dict(notes=scrub(edit.get('reply')), editor_error=editor_error, editor_cost=_usage(edit).get('cost'),
                   scrubbed=_scrub_folder(cand))   # before the candidate run, so what is tested is what is kept
    report |= dict(changed=_files(cand) != _files(current), problems=check_learned(cand))
    if capped(): return report
    if report['changed'] and not report['problems'] and not report['editor_error']:
        try: c = run('candidate-run', cand)
        except RuntimeError as e: c = None; report['candidate_error'] = str(e)
        if c is not None:
            report['candidate'] = _summary(c)
            if capped(): return report
            if better(c, bases):   # a cheaper route counts only if the answer is as good
                try: report['adopted'], report['judge'] = judge_reply(prompt, c, bases, work/'judge', editor, exe, original)
                except RuntimeError as e: report['judge_error'] = str(e)
            if report['adopted'] and learned is not None:
                routes = ', '.join(str(b['route']) for b in report['bases'])
                msg = f"{LESSON}: {' '.join(prompt.split())[:60]}\n\n{report['notes']}\n\nRoute cost (findings, failed calls, calls): base {routes}; candidate {report['candidate']['route']}."
                try: report['pending'] = keep_lesson(learned, lesson_entry(prompt, report['notes']), msg)
                except ValueError as e: report['adopted'], report['problems'] = False, [str(e)]
    (work/'report.json').write_text(json.dumps(report, indent=1, default=str))
    return report

In [ ]:
FAKE = r'''#!/usr/bin/env python3
"A stand-in for `ramabana`: the editor writes AGENTS.md; a run takes one call once AGENTS.md exists, three before."
import json, sys
from pathlib import Path
a = sys.argv[1:]
roots, cfg, prompt = a[a.index('--root') + 1].split(','), Path(a[a.index('--cfg') + 1]), a[-1]
ls = dict(tool='ls', ok=True, args={'path': ''})
if prompt.startswith('You improve'):
    Path(roots[0], 'AGENTS.md').write_text('Read file sizes from ls.\n')
    acts, reply = [dict(tool='create_file', ok=True, args={})], 'base-1 call 1: wrote AGENTS.md'
elif prompt.startswith('A person asked'): acts, reply = [], 'YES\nIt names the same file and size.'
elif (cfg/'learned'/'AGENTS.md').exists(): acts, reply = [ls], 'big.py 50'
else: acts, reply = [ls, dict(tool='run_shell', ok=False, args={'command': 'stat big.py'}), ls], 'big.py 50'
(cfg/'agent-history.jsonl').write_text(json.dumps(dict(prompt=prompt, activity=acts, reply=reply, usage={'cost': 0})) + '\n')
'''
with TemporaryDirectory() as d:
    d = Path(d)
    exe, repo = d/'fake-ramabana', d/'repo'
    exe.write_text(FAKE); exe.chmod(0o755); repo.mkdir()
    def git(*a): return subprocess.run(['git', '-C', str(repo), *a], check=True, capture_output=True, text=True).stdout.strip()
    git('init', '-q'); (repo/'big.py').write_text('x = 1\n'); git('add', '.')
    git('-c', 'user.name=t', '-c', 'user.email=t@example.com', 'commit', '-qm', 'one')
    start = dict(repo=str(repo), head=git('rev-parse', 'HEAD'), dirty=[])
    r = improve_round('sizes of the big files', start, d/'work', learned=d/'learned', exe=str(exe))
    test_eq((r['bases'][0]['route'], r['candidate']['route'], r['adopted']), ((0, 1, 3), (0, 0, 1), True))
    assert r['judge'].startswith('YES')
    test_eq(subprocess.run(['git', '-C', str(d/'work'/'base-1'/'repo'), 'remote'], capture_output=True, text=True).stdout, '')   # nothing can push back
    test_eq(sorted(_files(d/'learned')), [])   # kept as pending: HEAD, which Ramabana reads, is unchanged
    dated = f'dhrona-{datetime.now():%Y-%m-%d}.md'
    test_eq(_git(d/'learned', 'ls-tree', '-r', '--name-only', r['pending']), dated)
    assert 'wrote AGENTS.md' in _git(d/'learned', 'show', f"{r['pending']}:{dated}")   # the editor's notes, not a diff
    test_eq(sorted(_files(d/'work'/'candidate')), ['AGENTS.md'])   # the runs sit beside the candidate, never in it
    assert (d/'work'/'report.json').exists()
    assert not (d/'work'/'evidence'/'original.md').exists()
    real = dict(prompt='sizes of the big files', reply='big.py 50', activity=[dict(tool='run_shell', ok=True, args={'command': 'stat -f %z big.py'}, detail='50')])
    r = improve_round('sizes of the big files', start, d/'w7', exe=str(exe), original=real)
    assert 'stat -f %z big.py' in (d/'w7'/'evidence'/'original.md').read_text()   # the editor reads what the person's session ran
    test_fail(lambda: improve_round('x', start, d/'work', exe=str(exe)), contains='exists')
    test_fail(lambda: improve_round('x', start | dict(dirty=['big.py']), d/'w2', exe=str(exe)), contains='uncommitted')
    test_fail(lambda: improve_round('x', start, d/'w3', approve='auto', exe=str(exe)), contains='approve')
    doubting = d/'doubting-ramabana'
    doubting.write_text(FAKE.replace("'YES\\nIt names", "'NO\\nIt asks instead of naming")); doubting.chmod(0o755)
    r = improve_round('sizes of the big files', start, d/'w5', learned=d/'learned-5', exe=str(doubting))
    test_eq((r['candidate']['route'], r['adopted']), ((0, 0, 1), False))   # a cheaper route that answers worse is not kept
    assert r['judge'].startswith('NO') and not (d/'learned-5').exists()
    mute = d/'mute-ramabana'
    mute.write_text(FAKE.replace("'YES\\nIt names the same file and size.'", "''")); mute.chmod(0o755)
    r = improve_round('sizes of the big files', start, d/'w6', exe=str(mute))
    assert 'no answer' in r['judge_error'] and not r['adopted']
    assert all(map(_says_yes, ['YES', '**YES**', 'yes: it names the file'])) and not any(map(_says_yes, ['YESTERDAY', 'NO', 'Answer C: YES']))
    dead = d/'dead-ramabana'
    dead.write_text(FAKE.replace("usage={'cost': 0}", "usage={'turns': 0}")); dead.chmod(0o755)
    r = improve_round('sizes of the big files', start, d/'w4', exe=str(dead))
    assert 'no answer' in r['base_error'] and 'big.py 50' in r['base_error'] and not (d/'w4'/'editor').exists()   # no editor runs on empty evidence
    paid = d/'paid-ramabana'
    paid.write_text(FAKE.replace("usage={'cost': 0}", "usage={'cost': 1}")); paid.chmod(0o755)   # each run records a dollar
    r = improve_round('sizes of the big files', start, d/'w8', exe=str(paid), limit=1.5)
    assert '$2.00' in r['cost_limit'] and len(r['bases']) == 2 and not r['adopted'] and not (d/'w8'/'editor').exists()   # the bases passed the limit: no editor runs
    r = improve_round('sizes of the big files', start, d/'w9', exe=str(paid), limit=3.5)
    assert 'candidate' in r and 'judge' not in r and 'cost_limit' in r and not r['adopted']   # the candidate run passed it: no judge runs
    r = improve_round('sizes of the big files', start, d/'w10', exe=str(paid))
    test_eq((r['adopted'], 'cost_limit' in r, ROUND_LIMIT), (True, False, 8))   # a round of $5 is under the default
    full = d/'learned-full'; full.mkdir(); (full/'dhrona-2026-01-01.md').write_text('y' * DATED_MAX); learned_repo(full)
    r = improve_round('sizes of the big files', start, d/'w11', learned=full, exe=str(paid))
    assert not r['adopted'] and 'pending' not in r and f'past the {DATED_MAX:,}' in r['problems'][0], r   # the cap's reason is in the report

## Training on past sessions

Older sessions have no `start.json`, so they record no starting commit, and most record no folder either. `session_start` works both out. It places a turn in the repository that most of its absolute tool paths fall in, or else the first git repository among `roots`. Its `head` is that repository's last commit before the turn's `at`, which may be epoch seconds or an ISO date; an unreadable `at` leaves the turn unplaced. `dirty` is None because nobody knows what was uncommitted at that time.

`trainable` picks the turns worth replaying. It takes the first turn of each session when the request is at least 40 characters long, carried no attachments, ended complete and without an error, made between 1 and `max_calls` calls, and cost something, meaning at least one finding or failed call. An attached file belonged only to that session, so a replay could not see it. It also drops a turn that made a commit, because `at` marks the end of the turn and the commit found for its start would already contain that commit. A request asked in several sessions is kept once, from the cheapest, and the result is ordered by number of calls, fewest first. `trainable_sessions` runs `trainable` over Ramabana's own history and session index, takes each session's start from its `start.json` entry for the repository the work used, and drops a session whose entry had uncommitted changes. The entry is matched on the resolved path, because Ramabana writes resolved paths and a tool call may name a path through a link. An unreadable `start.json` or session index, such as one a crash cut short, counts as missing.

In [ ]:
#| export
def request_text(prompt):
    "The person's request inside `<user-request>` tags, else the whole prompt."
    found = re.findall(r'<user-request>\n?(.*?)\n?</user-request>', str(prompt or ''), re.S)
    return (found[-1] if found else str(prompt or '')).strip()

def _toplevel(path):
    "The git repository holding the absolute `path`, after `~` expansion, or None."
    p = Path(path).expanduser()
    if not p.is_absolute(): return None
    return next((str(d) for d in [p, *p.parents] if (d/'.git').exists()), None)

def _when(at):
    "A history row's `at`, an epoch in seconds or an ISO date, as an ISO date git understands; None when unreadable."
    try: return datetime.fromtimestamp(float(at), timezone.utc).isoformat()
    except (TypeError, ValueError, OverflowError, OSError): pass
    try: return datetime.fromisoformat(str(at)).isoformat()
    except ValueError: return None

def session_start(turn, roots=()):
    "`{repo, head, dirty, inferred}` for replaying `turn`'s session, or None when nothing places it in a repository."
    hits = Counter()
    for a in turn.get('activity') or []:
        for k in ('path', 'cwd', 'file', 'root'):
            if (r := _toplevel(str(tool_args(a).get(k) or ''))): hits[r] += 1
    repo = hits.most_common(1)[0][0] if hits else next((r for r in map(_toplevel, roots) if r), None)
    if repo is None or (when := _when(turn.get('at'))) is None: return None
    head = subprocess.run(['git', '-C', repo, 'rev-list', '-1', f'--before={when}', 'HEAD'], stdin=subprocess.DEVNULL, capture_output=True, text=True).stdout.strip()
    return dict(repo=repo, head=head, dirty=None, inferred=True) if head else None

def trainable(turns, roots_of=None, max_calls=40):
    "The first turn of each session that can seed a round, shortest first: `{session, prompt, model, start, route, turn}`."
    first = {}
    for t in turns:
        if t.get('session'): first.setdefault(t['session'], t)
    found = []
    for sid, t in first.items():
        req, cost = request_text(t.get('prompt')), route_cost(t)
        if len(req) < 40 or '<attachments>' in str(t.get('prompt')) or t.get('error') or t.get('state', 'complete') != 'complete': continue   # an attachment is a file only that session had
        if not 0 < cost[2] <= max_calls or cost[:2] == (0, 0): continue
        if any(a.get('tool') == 'git_commit' for a in t.get('activity') or []): continue    # `at` is the turn's end, so its start would hold this commit
        if (start := session_start(t, (roots_of or {}).get(sid) or ())) is None: continue
        found.append(dict(session=sid, prompt=req, model=t.get('model'), start=start, route=cost, turn=t))
    out, seen = [], set()
    for s in sorted(found, key=lambda s: s['route'][2]):
        if s['prompt'] not in seen: seen.add(s['prompt']); out.append(s)
    return out

def _json(p):
    "The JSON in `p`, or None when it is missing or unreadable: a write a crash cut short."
    try: return json.loads(Path(p).read_text())
    except (OSError, ValueError): return None

def _same(a, b):
    "Whether two paths name one folder; Ramabana writes resolved paths and a tool call's path may go through a link."
    return bool(a) and bool(b) and Path(a).expanduser().resolve() == Path(b).expanduser().resolve()

def trainable_sessions(cfg='~/.config/ramabana', max_calls=40):
    "`trainable` over Ramabana's history and session index in `cfg`, with each session's own `start.json` where it has a clean one."
    cfg = Path(cfg).expanduser()
    idx = _json(cfg/'agent-sessions.json')
    roots = {s: v.get('roots') for s, v in idx.get('sessions', {}).items()} if isinstance(idx, dict) else {}
    out = []
    for s in trainable(read_history(cfg/'agent-history.jsonl'), roots, max_calls):
        recs = _json(cfg/'runs'/s['session']/'start.json')
        rec = next((r for r in recs if isinstance(r, dict) and _same(r.get('repo'), s['start']['repo'])), None) if isinstance(recs, list) else None
        if rec is not None:
            if rec.get('dirty'): continue
            s['start'] = rec | dict(inferred=False)
        out.append(s)
    return out

In [ ]:
import os
from datetime import datetime, timezone
with TemporaryDirectory() as d:
    repo = Path(d)/'repo'; repo.mkdir()
    def git(*a, when=None):
        env = os.environ | ({'GIT_AUTHOR_DATE': when, 'GIT_COMMITTER_DATE': when} if when else {})
        return subprocess.run(['git', '-C', str(repo), *a], check=True, capture_output=True, text=True, env=env).stdout.strip()
    git('init', '-q')
    for n, when in [(1, '2026-01-01T00:00:00Z'), (2, '2026-03-01T00:00:00Z')]:
        (repo/'a.py').write_text(f'x = {n}\n'); git('add', '.')
        git('-c', 'user.name=t', '-c', 'user.email=t@example.com', 'commit', '-qm', f'c{n}', when=when)
    january = git('rev-list', '--max-parents=0', 'HEAD')
    t = {'session': 's1', 'at': datetime(2026, 2, 1, tzinfo=timezone.utc).timestamp(), 'model': 'claude-opus-5-5',
         'prompt': '<user-request>\nwhere is x defined, and which files read it? list them\n</user-request>',
         'activity': [dict(tool='grep', ok=True, args={'pattern': 'x', 'path': str(repo/'a.py')}),
                      dict(tool='run_shell', ok=False, args={'command': 'cat a.py'})]}
    test_eq((session_start(t)['repo'], session_start(t)['head']), (str(repo), january))   # the last commit before the turn
    test_eq(session_start(t | {'at': '2026-02-01T00:00:00+00:00'})['head'], january)        # an ISO `at` reads the same
    test_eq((session_start(t | {'activity': []}), session_start(t | {'at': None}), _toplevel('relative/dir')), (None, None, None))
    test_eq(session_start(t | {'activity': []}, roots=[str(repo)])['repo'], str(repo))
    attached = t | {'session': 's3', 'prompt': '<user-request>\nwhat does this screenshot show about the layout?\n</user-request>\n<attachments>\nimage /tmp/paste.png\n</attachments>'}
    cheaper = t | {'session': 's5', 'activity': [dict(tool='grep', ok=False, args={'pattern': 'x', 'path': str(repo/'a.py')})]}
    committed = t | {'session': 's6', 'prompt': 'where is x defined, and who imports it? list each file please',
                     'activity': t['activity'] + [dict(tool='git_commit', ok=True, args={'message': 'm'})]}
    got = trainable([t, t | {'prompt': 'a later turn in the same session'}, t | {'session': 's2', 'prompt': 'ok'},
                     t | {'session': 's4'}, attached, cheaper, committed])
    test_eq([(s['session'], s['prompt'], s['model']) for s in got], [('s5', 'where is x defined, and which files read it? list them', 'claude-opus-5-5')])
    test_eq(got[0]['turn']['session'], 's5')   # the session's own first turn, for the editor and the judge
    # s4 and s5 ask what s1 asked and s5 costs least; s3 names a file only its session had; s6's start would hold its own commit
    cfg = Path(d)/'cfg'; (cfg/'runs'/'s1').mkdir(parents=True)
    (cfg/'agent-history.jsonl').write_text(json.dumps(t) + '\n')
    rec = [dict(repo='/elsewhere', head='abc', dirty=[]), dict(folder=str(repo), repo=str(repo), head=january, dirty=[])]
    (cfg/'runs'/'s1'/'start.json').write_text(json.dumps(rec))
    test_eq([(s['start']['head'], s['start']['inferred']) for s in trainable_sessions(cfg)], [(january, False)])   # the entry for the repository the work used
    rec[1]['dirty'] = ['a.py']; (cfg/'runs'/'s1'/'start.json').write_text(json.dumps(rec))
    test_eq(trainable_sessions(cfg), [])

In [ ]:
import time
with TemporaryDirectory() as d:
    d = Path(d); repo = d/'repo'; repo.mkdir()
    def git(*a): return subprocess.run(['git', '-C', str(repo), *a], check=True, capture_output=True, text=True).stdout.strip()
    git('init', '-q'); (repo/'a.py').write_text('x = 1\n'); git('add', '.')
    git('-c', 'user.name=t', '-c', 'user.email=t@example.com', 'commit', '-qm', 'c')
    (link := d/'link').symlink_to(repo)
    t = {'session': 's1', 'at': time.time() + 60, 'model': 'm', 'prompt': 'where is x defined, and which files read it? list them',
         'activity': [dict(tool='grep', ok=True, args={'path': str(link/'a.py')}), dict(tool='run_shell', ok=False, args={'command': 'cat a.py'})]}
    test_eq(trainable([t | {'state': 'cancelled'}]), [])   # a turn the person stopped seeds nothing
    test_eq(len(trainable([t | {'state': 'complete'}])), 1)
    cfg = d/'cfg'; (cfg/'runs'/'s1').mkdir(parents=True); (cfg/'agent-history.jsonl').write_text(json.dumps(t) + '\n')
    rec = dict(folder=str(repo), repo=str(repo.resolve()), head=git('rev-parse', 'HEAD'), dirty=['a.py'])   # Ramabana resolves the path
    (cfg/'runs'/'s1'/'start.json').write_text(json.dumps([rec]))
    test_eq(trainable_sessions(cfg), [])   # the record for the work's repository, reached through a link, says it began dirty
    for bad in ('[{"repo', '{}', 'null', '[1]'):
        (cfg/'runs'/'s1'/'start.json').write_text(bad)
        test_eq([s['start']['inferred'] for s in trainable_sessions(cfg)], [True])   # an unreadable record is no record
    (cfg/'agent-sessions.json').write_text('{"sess')
    test_eq(len(trainable_sessions(cfg)), 1)

## Batches

`improve_batch` runs `improve_round` on up to `n` sessions in turn, in the order `trainable_sessions` gives. Every round starts from the learned folder's HEAD, so uncommitted text of a person's never reaches a replay. Each kept lesson waits on its own `pending/` branch, so each can be accepted or rejected alone. With `cfg` set, the batch first expires what went `days` days unused (`expire_learned`, listed in `expired.json`) and gives every round the lessons of the last `days` days (`recent_lessons`). With `compact_at` set, it then proposes a compaction of `AGENTS.md` when the file is over that fraction of `LEARNED_MAX` (see Compaction), and writes the result to `compaction.json`. `models` keeps the sessions recorded with those models, and `skip` leaves sessions out by id. A round that fails, for example because its model cannot start or its repository is gone, becomes a row with `error`, and the batch goes on. `summary.json` in `work` holds every row so far, and `[]` when no session matched. An `approve` other than `ask` or `edits` raises before any round runs. Uncommitted files in the learned folder never stop a batch.

Each row has a `cost`: the dollars recorded by every run in the round's folder (`round_cost`), meaning the base runs, the editor, the candidate and the judge, whether or not the round finished. A run killed at its timeout, or one whose model reports no cost, adds nothing. With `budget` set, no new round starts once the rounds so far have cost that much, so a batch can overshoot its budget by the cost of its last round. A batch with a learned folder holds `learned_lock` on it for its whole run, and a second batch on the same folder, from the command line or a scheduled beat, raises `RuntimeError` before it creates anything. The lock is a file beside the folder, and the operating system releases it when the batch's process ends, even in a crash. `dhrona-improve` runs a batch from the command line. It requires a learned folder, and prints one line per round, with the pending branches and `REVIEW`, which says to review them with `/learned` in Ramabana or `ramabana-learned` in a shell.

In [ ]:
#| export
#: Where a pending lesson or compaction is reviewed. `ramabana-learned` takes its subcommand as `--cmd`, and lists with none.
REVIEW = 'review with: /learned in Ramabana, or ramabana-learned in a shell'

class BatchBusy(RuntimeError): 'Another batch holds the learned folder.'

@contextmanager
def learned_lock(learned):
    "Hold `learned`'s lock for the block, or raise `RuntimeError` at once when another batch holds it."
    learned = Path(learned)
    learned.parent.mkdir(parents=True, exist_ok=True)
    with open(learned.parent/f'.{learned.name}.lock', 'w') as f:   # beside the folder: inside it, `git status` would count it as a change
        try:
            if os.name == 'nt': import msvcrt; msvcrt.locking(f.fileno(), msvcrt.LK_NBLCK, 1)
            else: import fcntl; fcntl.flock(f, fcntl.LOCK_EX | fcntl.LOCK_NB)
        except OSError: raise BatchBusy(f'another batch is using {learned}') from None
        yield

def _refused(work, e):
    "Record why batch folder `work` could not start, as its `night-failed` marker and the one row of its `summary.json`; returns the text."
    msg = f'{type(e).__name__}: {e}'
    (work/FAILED).write_text(msg)
    (work/'summary.json').write_text(json.dumps([dict(error=msg)]))
    return msg

def improve_batch(
    work,              # a new folder for one round folder per session and `summary.json`
    learned=None,      # the learned folder; every round starts from its HEAD, and a kept lesson waits on a `pending/` branch
    sessions=None,     # `trainable_sessions()` when None
    n=10,              # rounds to run
    model=None,        # replay every session with this model; None uses each session's own
    editor=None,       # the model that edits each candidate
    models=None,       # keep only sessions recorded with one of these models
    skip=(),           # session ids to leave out
    k=2,               # base runs per round
    approve='edits',   # `ask` refuses every write in a replay
    exe=RAMABANA,      # the Ramabana command a round runs
    read_outside=True, # replays read outside their clone
    budget=None,       # dollars; no round starts once the rounds so far have cost this much
    cfg=None,          # Ramabana's config folder: its history gives the lessons and what was used; None reads neither
    days=30,           # lessons from this many days; what Dhrona proposed and nothing used in them expires
    compact_at=None,   # a fraction of `LEARNED_MAX`: past it, a compaction of `AGENTS.md` is proposed before the rounds
    timeout=1800,      # seconds before a replay is abandoned
):
    "Run `improve_round` on up to `n` sessions in turn; returns one row per round, also written to `work/summary.json`."
    work = Path(work)
    if work.exists(): raise FileExistsError(f'{work} exists; a batch writes into a new folder')
    if approve not in ('ask', 'edits'): raise ValueError(f"approve is 'ask' or 'edits', not {approve!r}: a replay never runs shell commands")
    if isinstance(models, str): models = [m.strip() for m in models.split(',') if m.strip()]   # a schedule's payload may hold them as text
    with learned_lock(learned) if learned is not None else nullcontext():
        work.mkdir(parents=True)   # before any check that can raise, so a refused night leaves its marker
        try:
            expired = expire_learned(learned, cfg, days) if learned is not None and cfg is not None else []
            compaction = _due_compaction(learned, compact_at, _editor(editor, exe)) if learned is not None else None
            lessons = recent_lessons(cfg, days) if cfg is not None else []
            picked = [s for s in (trainable_sessions() if sessions is None else sessions)
                      if s['session'] not in skip and (models is None or s['model'] in models)][:n]
        except Exception as e: _refused(work, e); raise
        rows, spent = [], (compaction or {}).get('cost') or 0   # the compaction's editor turn counts against `budget`
        (work/'summary.json').write_text('[]')
        (work/'expired.json').write_text(json.dumps(expired))
        (work/'compaction.json').write_text(json.dumps(compaction))
        for s in picked:
            if budget is not None and spent >= budget: break
            row = dict(session=s['session'], prompt=s['prompt'], model=model or s['model'])
            try:
                r = improve_round(s['prompt'], s['start'], work/s['session'], learned, row['model'], editor, k, approve, exe, read_outside,
                                  original=s.get('turn'), lessons=lessons, timeout=timeout)
                row |= dict(adopted=r['adopted'], pending=r.get('pending'), changed=r.get('changed'), problems=r.get('problems'),
                            bases=[b['route'] for b in r['bases']], candidate=(r.get('candidate') or {}).get('route'), judge=r.get('judge'),
                            error=r.get('base_error') or r.get('editor_error') or r.get('candidate_error') or r.get('judge_error') or r.get('cost_limit') or None)
            except (RuntimeError, ValueError, OSError, subprocess.CalledProcessError) as e: row['error'] = f'{type(e).__name__}: {e}'
            row['cost'] = round_cost(work/s['session'])
            spent += row['cost']
            rows.append(row)
            (work/'summary.json').write_text(json.dumps(rows, indent=1, default=str))
    return rows

@call_parse
def improve_cli(
    work:str,                       # a new folder for this batch
    learned:str,                    # the learned folder to start from; kept lessons wait on its `pending/` branches
    n:int=10,                       # rounds to run
    k:int=2,                        # base runs per round
    model:str=None,                 # replay every session with this model instead of its own
    editor:str=None,                # the model that edits each candidate
    models:str='',                  # comma-separated: replay only sessions recorded with these models
    skip:str='',                    # comma-separated session ids to leave out
    approve:str='edits',            # `ask` refuses every write in a replay
    cfg:str='~/.config/ramabana',   # the Ramabana config folder whose history is read
    max_calls:int=40,               # replay only sessions whose first turn made at most this many calls
    exe:str=RAMABANA,               # the Ramabana command a round runs
    read_outside:bool=True,         # --no-read-outside keeps replays' reads inside their clone
    budget:float=None,              # dollars: no round starts once the batch has cost this much
    timeout:int=1800,               # seconds before a replay is abandoned
):
    "Run up to `n` improvement rounds on past Ramabana sessions and print one line per round, then what they cost."
    rows = improve_batch(work, learned, trainable_sessions(cfg, max_calls), n, model, editor, [m for m in models.split(',') if m] or None,
                         [s for s in skip.split(',') if s], k, approve, exe, read_outside, budget, cfg, timeout=timeout)
    if not rows: print('no session matched')
    for r in rows:
        if r.get('error'): out = r['error']
        elif r.get('adopted'): out = f"pending {r['pending']}"
        elif r.get('problems'): out = f"not kept: {r['problems'][0]}"
        elif not r.get('changed'): out = 'not kept: the editor changed nothing'
        elif r.get('judge'): out = 'not kept: the judge preferred a base answer'
        else: out = 'not kept: the candidate was no cheaper'
        print(f"{r['session']}  {r['model']}  {out}  {r['prompt'][:60]}")
    if rows: print(f"spent ${sum(r['cost'] for r in rows):.2f}")
    if any(r.get('adopted') for r in rows): print(REVIEW)

In [ ]:
#| export
TIMES = 'Lesson-times: '   #: the trailer in a compaction's message holding, per rewritten line, when its source lessons were last touched

def _key(line): return hashlib.sha1(line.strip().encode()).hexdigest()[:10]

def _times(learned, sha):
    "The `TIMES` trailer of commit `sha`, as line key to epoch seconds."
    for l in _git(learned, 'log', '-1', '--format=%B', sha).splitlines():
        if l.startswith(TIMES):
            try: return {k: int(v) for k, v in json.loads(l[len(TIMES):]).items()}
            except (ValueError, AttributeError): pass
    return {}

def _blame(learned, rel, rev='HEAD'):
    "`(time, subject)` for each line of `rel` at `rev`: its commit's time, or for a line a compaction wrote, its sources' newest time."
    out, cur, trailers = [], {}, {}
    for l in _git(learned, 'blame', '--line-porcelain', rev, '--', rel).splitlines():
        if l.startswith('\t'):
            t, s = int(cur.get('committer-time', 0)), cur.get('summary', '')
            if s.startswith(COMPACT): t = trailers.setdefault(cur['sha'], _times(learned, cur['sha'])).get(_key(l[1:]), t)
            out.append((t, s))
        elif re.fullmatch(r'[0-9a-f]{40}( \d+)+', l): cur = dict(sha=l[:40])
        else: k, _, v = l.partition(' '); cur[k] = v
    return out

def _blocks(lines):
    "Line indexes grouped into paragraphs, a fenced block never split."
    out, cur, fence = [], [], False
    for n, l in enumerate(lines):
        if l.strip().startswith('```'): fence = not fence
        if not l.strip() and not fence:
            if cur: out.append(cur); cur = []
        else: cur.append(n)
    return out + ([cur] if cur else [])

def expire_dated(learned, days=DATED_DAYS, now=None):
    "Remove each committed dated learning file whose date is over `days` days old, one commit per file as Dhrona; returns the names. Only that file's path is staged or committed, so other edits stay as they are."
    learned, gone = Path(learned), []
    today = datetime.fromtimestamp(now or time.time()).date()
    for name in _git(learned, 'ls-files', '--', 'dhrona-*.md').split('\n'):
        if not DATED.fullmatch(name) or (today - datetime.strptime(name[7:-3], '%Y-%m-%d').date()).days <= days: continue
        try: _git(learned, 'rm', '-q', '--', name); _git(learned, *_AS_DHRONA, 'commit', '-q', '-m', f'Expired {name} after {days} days', '--', name)
        except subprocess.CalledProcessError: continue   # a person edited it since the last commit
        gone.append(name)
    return gone

def expire_learned(learned, cfg='~/.config/ramabana', days=30, now=None):
    "Remove the dated learning files over `DATED_DAYS` old, and the skills and `AGENTS.md` paragraphs Dhrona proposed that nothing used or renewed in `days` days; returns what went."
    learned = Path(learned)
    if days is None or not (learned/'.git').exists(): return []
    dated = expire_dated(learned, now=now)
    if _uncommitted(learned): return dated
    cutoff = (now or time.time()) - days*86400
    recent = [t for t in _history(cfg) if _epoch(t.get('at')) >= cutoff]
    if not recent: return dated   # nobody worked, so nothing had the chance to be used
    used = ({str(tool_args(a).get('name') or '') for t in recent for a in t.get('activity') or [] if a.get('tool') == 'read_skill'}
            | {s for t in recent for s in (t.get('plan') or {}).get('skills') or []})
    stale = lambda ct, subj: ct < cutoff and _is_dhronas(subj)
    gone = []
    for d in sorted((learned/'skills').iterdir()) if (learned/'skills').is_dir() else []:
        if not d.is_dir() or d.name in used: continue
        ct, _, subj = _git(learned, 'log', '-1', '--format=%ct %s', '--', f'skills/{d.name}').partition(' ')
        if ct and stale(int(ct), subj): shutil.rmtree(d); gone.append(f'skills/{d.name}')
    if (p := learned/'AGENTS.md').exists() and _git(learned, 'ls-files', 'AGENTS.md'):
        lines, who = p.read_text().splitlines(), _blame(learned, 'AGENTS.md')
        drop = [b for b in _blocks(lines) if all(stale(*who[n]) for n in b)]
        if drop:
            cut = {n for b in drop for n in (b[0] - 1, *b) if n >= 0 and (n in b or not lines[n].strip())}   # with the blank line before it
            kept = '\n'.join(l for n, l in enumerate(lines) if n not in cut).strip()
            p.write_text(kept + '\n') if kept else p.unlink()
            gone += [f'AGENTS.md: {lines[b[0]][:60]}' for b in drop]
    if gone:
        paths = sorted({g.split(':')[0] for g in gone})
        _git(learned, 'add', '--', *paths); _git(learned, *_AS_DHRONA, 'commit', '-q', '-m', f'Expired after {days} days unused: ' + '; '.join(gone), '--', *paths)
    return dated + gone

`expire_learned` removes what Dhrona proposed and a person accepted once it has gone `days` days without use or renewal, and commits the removal so `git revert` brings it back. A skill counts as used when a session read it with `read_skill` or the turn loaded it, and renewed when a later accepted round changed it. `AGENTS.md` is in every briefing, so a paragraph there counts only as renewed when an accepted round last changed one of its lines; a fenced block is never split. Only commits whose message starts with `LESSON` can expire: what was there before Dhrona, and anything a person committed, stays. When nobody worked in the last `days` days, nothing expires, because nothing had the chance to be used. While the folder has uncommitted changes, skills and `AGENTS.md` paragraphs do not expire, because the removal is committed with the files it changed and must not stage a person's edit. It stages and commits only the paths it changed.

`expire_dated` removes each committed dated learning file over `DATED_DAYS` (30) days old, by the date in its name. It runs `git rm` and a commit on that one path as Dhrona, one commit a file, and touches no other path. It works on a folder with uncommitted edits. `expire_learned` calls it first, so a dated file expires even when nobody worked.

In [ ]:
with TemporaryDirectory() as d:
    d = Path(d); cfg, learned, cand = d/'cfg', d/'learned', d/'cand'
    (learned/'skills'/'mine').mkdir(parents=True); (learned/'skills'/'mine'/'SKILL.md').write_text('---\nname: mine\ndescription: x\n---\nhand\n')
    (learned/'AGENTS.md').write_text('A rule a person wrote.\n')
    learned_repo(learned)
    old = str(int(time.time()) - 40*86400)
    def lesson(files, msg, when):
        for rel, text in files.items(): (learned/rel).parent.mkdir(parents=True, exist_ok=True); (learned/rel).write_text(text)
        env = os.environ | {'GIT_COMMITTER_DATE': when, 'GIT_AUTHOR_DATE': when}
        subprocess.run(['git', '-C', str(learned), 'add', '-A'], check=True)
        subprocess.run(['git', '-C', str(learned), *_AS_DHRONA, 'commit', '-qm', msg], check=True, env=env)
    lesson({'skills/stale/SKILL.md': '---\nname: stale\ndescription: x\n---\nold\n', 'skills/read/SKILL.md': '---\nname: read\ndescription: x\n---\nold\n',
            'AGENTS.md': 'A rule a person wrote.\n\nAn old rule.\n```\ncode\n\nmore\n```\n'}, f'{LESSON}: old', old)
    lesson({'AGENTS.md': 'A rule a person wrote.\n\nAn old rule.\n```\ncode\n\nmore\n```\n\nA new rule.\n'}, f'{LESSON}: new', str(int(time.time())))
    cfg.mkdir()
    test_eq(expire_learned(learned, cfg), [])   # no session in the window: nothing had the chance to be used
    (cfg/'agent-history.jsonl').write_text(json.dumps(dict(session='s', at=time.time(), activity=[dict(tool='read_skill', ok=True, args={'name': 'read'})])) + '\n')
    gone = expire_learned(learned, cfg)
    test_eq(gone, ['skills/stale', 'AGENTS.md: An old rule.'])
    test_eq((learned/'AGENTS.md').read_text(), 'A rule a person wrote.\n\nA new rule.\n')
    test_eq(sorted(p.name for p in (learned/'skills').iterdir()), ['mine', 'read'])
    assert not _uncommitted(learned) and 'Expired after 30 days' in _git(learned, 'log', '-1', '--format=%s')

`expire_learned` also removes each dated learning file whose date is over `DATED_DAYS` (30) days old (`expire_dated`). It removes the file with `git rm` and commits that path alone as Dhrona, one commit a file. It stages and commits no other path, so it runs on a folder with uncommitted edits. A person's edit to a dated file that is not yet committed keeps the file.

In [ ]:
with TemporaryDirectory() as d:
    d = Path(d); learned = d/'learned'
    learned.mkdir(); (learned/'AGENTS.md').write_text('Read sizes from `ls`.\n')
    (learned/'dhrona-2026-08-01.md').write_text('# Dhrona lessons 2026-08-01\n'); (learned/'dhrona-2026-10-20.md').write_text('# Dhrona lessons 2026-10-20\n')
    learned_repo(learned)
    (learned/'AGENTS.md').write_text('a hand edit\n')
    (learned/'notes.txt').write_text('draft'); _git(learned, 'add', 'notes.txt')   # a person's staged file
    before = _git(learned, 'status', '--porcelain')
    gone = expire_learned(learned, d/'cfg', now=datetime(2026, 10, 26).timestamp())
    test_eq(gone, ['dhrona-2026-08-01.md'])
    test_eq(_git(learned, 'show', '--name-only', '--format=%an', 'HEAD'), 'Dhrona\n\ndhrona-2026-08-01.md')   # one path, authored by Dhrona
    test_eq(_git(learned, 'status', '--porcelain'), before)   # the edit is still unstaged, the staged file still staged
    test_eq((learned/'AGENTS.md').read_text(), 'a hand edit\n')
    test_eq(sorted(_files(learned)), ['AGENTS.md', 'dhrona-2026-10-20.md', 'notes.txt'])
before

## Compaction

Accepted lessons add up, and `ramabana-learned accept` refuses one that would take `AGENTS.md` past `LEARNED_MAX`. `compact_learned` proposes a shorter file. git blame splits HEAD's `AGENTS.md` into paragraphs Dhrona proposed, where every line came from a commit whose message starts with `LESSON` or `COMPACT`, and the rest, which a person wrote or which was there before Dhrona. The editor model rewrites only Dhrona's paragraphs. It merges lessons that overlap, drops duplicates and lessons a later one replaced, and keeps fenced code blocks whole, aiming at `target` characters (`COMPACT_TARGET`, 1,200) for the whole file. A person's paragraph stands in the file as a marker line such as `[kept 1]`, with its text given separately for context, and the marker becomes the original paragraph again, byte for byte. The prompt and the rewrite are scrubbed.

A rewrite is refused, and nothing is committed, when it drops, moves, repeats or edits a marker, drops a fenced block, is no shorter than the file it started from, or passes `LEARNED_MAX`. One that passes waits on a `pending/` branch like a kept candidate, so a person reviews it with `ramabana-learned`. Its commit message starts with `COMPACT`, so what it wrote counts as Dhrona's own: it expires as a lesson does, and a later compaction may rewrite it. Accepting it does not renew the lessons it rewrote. Each rewritten paragraph takes the newest time of the lessons it draws on, matched by shared words, or of all of them when none matches, and the commit's `TIMES` trailer records that time for each of its lines. `_blame` reads the trailer, so a paragraph merged from two stale lessons expires on their schedule, not 30 days after the accept.

`onto` folds in a pending candidate that `ramabana-learned accept` refused for size. The file that candidate would make is compacted, the compaction holds the candidate's other files too, and its `INCLUDES` trailer names the candidate, so accepting the compaction accepts the candidate and drops its branch. Ramabana's `accept` then moves every other pending candidate onto the new HEAD where it applies cleanly, and marks the rest stale.

`compact_learned` takes `learned_lock`. It refuses while a batch runs, when `AGENTS.md` holds no paragraph Dhrona proposed, and when a compaction is already pending. Uncommitted changes in the folder do not stop it: it works from HEAD and leaves the working tree alone. It returns `{ok, pending, before, after, cost}`, where `before` and `after` are character counts and `cost` the editor's dollars, or `{ok: False, error}`. The editor runs as the judge does: one headless Ramabana turn with the `editor` model, in an empty folder. A test passes `chat`, a function from prompt to reply.

In [ ]:
#| export
INCLUDES = 'Includes: '   #: the trailer naming the pending candidate a compaction folded in
COMPACT_TARGET = 1200   #: characters a compaction aims the whole `AGENTS.md` at
COMPACTION = r"""You shorten the always-on rules of a coding agent, Ramabana, without losing any of them.

Ramabana puts the `AGENTS.md` below in every briefing. It is {n} characters. Dhrona, a tool that learns from replays of past sessions, wrote its paragraphs one lesson at a time, so lessons overlap and repeat. A person wrote each paragraph shown as a `[kept N]` line. Those read as follows, for context only:

{kept}

Rewrite Dhrona's paragraphs.
- Merge lessons that overlap, and drop duplicates.
- When two lessons disagree, keep the later one, which sits lower in the file.
- Drop a lesson that a kept paragraph already says.
- Copy every fenced code block whole and unchanged.
- Add no rule, and change no rule's meaning.
- Aim at {target} characters for the whole file, kept paragraphs included. Never pass {mx}.

Reply with the new file and nothing else. Keep every `[kept N]` line, in the same order, alone on its line with a blank line before and after it.

<file>
{file}
</file>"""
_KEPT = re.compile(r'\[kept (\d+)\]')

def _beside_python():
    "The `ramabana` installed beside this interpreter, else the name for `PATH` to find."
    return shutil.which(RAMABANA, path=str(Path(sys.executable).parent)) or RAMABANA


def _paragraphs(learned, rev='HEAD'):
    "`AGENTS.md` at `rev` as `(text, Dhrona's, time)` per paragraph: Dhrona's when each line came from a commit Dhrona proposed; `time` its newest line's."
    lines, who = _show(learned, rev).splitlines(), _blame(learned, 'AGENTS.md', rev)
    return [('\n'.join(lines[n] for n in b), all(_is_dhronas(who[n][1]) for n in b), max(who[n][0] for n in b)) for b in _blocks(lines)]

def _words(t): return set(re.findall(r'[a-z0-9_]{4,}', t.lower()))

def _source_time(text, sources):
    "The newest time among `sources`, `(text, time)` pairs, whose words `text` carries at least half of; the newest of all when none."
    w = _words(text)
    return max([t for s, t in sources if (sw := _words(s)) and 2*len(w & sw) >= len(sw)] or [t for _, t in sources])

def compaction_prompt(before, paras, target=COMPACT_TARGET):
    "The scrubbed prompt asking the editor to compact `before`, whose `paras` come from `_paragraphs`, with a person's paragraphs as `[kept N]`."
    kept, it = [t for t, mine, _ in paras if not mine], iter(range(1, len(paras) + 1))
    file = '\n\n'.join(t if mine else f'[kept {next(it)}]' for t, mine, _ in paras)
    ks = '\n\n'.join(f'[kept {i}]\n{t}' for i, t in enumerate(kept, 1)) or 'There are none.'
    return scrub(COMPACTION.format(n=len(before), kept=ks, file=file, target=target, mx=LEARNED_MAX))

def _fences(text): return re.findall(r'^[ \t]*```.*?^[ \t]*```[^\n]*$', text, re.M | re.S)

def compacted(reply, before, paras):
    "The file a compaction `reply` proposes, each `[kept N]` restored and the rest scrubbed; raises `ValueError` saying why it is refused."
    kept = [t for t, mine, _ in paras if not mine]
    s = re.sub(r'^\s*<file>|</file>\s*$', '', str(reply or '').replace('\r\n', '\n')).strip()
    if (m := re.fullmatch(r'```[\w-]*\n(.*)\n```', s, re.S)): s = m[1]   # a fence around the whole reply; unwrapping a real pair loses a fence, which is refused
    if not s: raise ValueError('the editor gave no file')
    lines, out, seen = s.splitlines(), [], []
    for b in _blocks(lines):
        t = '\n'.join(lines[n] for n in b)
        if (m := _KEPT.fullmatch(t.strip())) and 0 < int(m[1]) <= len(kept): seen.append(int(m[1])); out.append(kept[int(m[1]) - 1])
        elif _KEPT.search(t): seen.append(None)
        else: out.append(scrub(t))
    if seen != list(range(1, len(kept) + 1)): raise ValueError('the rewrite moved, dropped, repeated or changed a paragraph a person wrote')
    text = '\n\n'.join(out) + '\n'
    if (lost := [f for f in _fences(before) if f not in text and scrub(f) not in text]):
        raise ValueError(f'the rewrite drops the fenced block that starts {lost[0].strip().splitlines()[0]}')
    if len(text) >= len(before): raise ValueError(f'the rewrite is {len(text):,} characters, no shorter than the {len(before):,} it started from')
    if len(text) > LEARNED_MAX: raise ValueError(f'the rewrite is {len(text):,} characters, past the {LEARNED_MAX:,} Dhrona allows')
    return text

def _pending_compaction(learned):
    "The `pending/` branch holding a compaction of `learned`, or None."
    refs = _git(learned, 'for-each-ref', '--format=%(refname:short)%00%(subject)', f'refs/heads/{PENDING}')
    return next((r for r, _, s in (l.partition('\0') for l in refs.splitlines()) if s.startswith(COMPACT)), None)

def _editor(editor=None, exe=None):
    "A `chat(prompt)` that asks `editor` in one headless Ramabana turn in an empty folder, as a round asks its judge; `chat.cost` sums the dollars."
    def chat(prompt):
        with tempfile.TemporaryDirectory() as d: t = run_turn(prompt, [d], Path(d)/'cfg', editor, exe=exe or _beside_python())
        chat.cost += _usage(t).get('cost') or 0
        if not delivered(t): raise RuntimeError(f"the editor gave no answer: {t.get('error') or t.get('reply') or ''}")
        return t['reply']
    chat.cost = 0
    return chat

def _compact(learned, target, chat, onto=None):
    "`compact_learned` with the lock already held."
    learned, rev = Path(learned), PENDING + onto if onto else 'HEAD'
    if not (learned/'.git').exists(): return dict(ok=False, error=f'{learned} has no committed AGENTS.md to compact')
    if onto and subprocess.run(['git', '-C', str(learned), 'rev-parse', '--verify', '-q', f'refs/heads/{rev}'], stdin=subprocess.DEVNULL, capture_output=True).returncode:
        return dict(ok=False, error=f'nothing is pending as {onto}')
    if not _git(learned, 'ls-tree', '--name-only', rev, 'AGENTS.md'): return dict(ok=False, error=f'{learned} has no committed AGENTS.md to compact')
    if (b := _pending_compaction(learned)): return dict(ok=False, error=f'a compaction is already pending as {b.removeprefix(PENDING)}; accept or reject it first')
    before, paras = _show(learned, rev), _paragraphs(learned, rev)
    if not any(mine for _, mine, _ in paras): return dict(ok=False, error='AGENTS.md holds nothing Dhrona proposed, and a compaction rewrites only that')
    try: text = compacted(chat(compaction_prompt(before, paras, target)), before, paras)
    except (ValueError, RuntimeError, OSError, subprocess.SubprocessError) as e: return dict(ok=False, error=str(e), cost=getattr(chat, 'cost', 0))
    srcs, kept, times, lines = [(t, w) for t, mine, w in paras if mine], {t for t, mine, _ in paras if not mine}, {}, text.splitlines()
    for b in _blocks(lines):
        if (bt := '\n'.join(lines[n] for n in b)) in kept: continue
        for n in b: times[_key(lines[n])] = max(times.get(_key(lines[n]), 0), _source_time(bt, srcs))   # expires by its sources, not by this commit
    msg = (f"{COMPACT}{f' with {onto}' if onto else ''} ({len(before):,} → {len(text):,} characters)\n\n"
           + (f'{INCLUDES}{onto}\n' if onto else '') + TIMES + json.dumps(times, separators=(',', ':')))
    with tempfile.TemporaryDirectory() as d:
        tarfile.open(fileobj=io.BytesIO(subprocess.run(['git', '-C', str(learned), 'archive', rev], stdin=subprocess.DEVNULL, capture_output=True, check=True).stdout)).extractall(cand := Path(d)/'cand', filter='data')
        (cand/'AGENTS.md').write_text(text)
        branch = keep(cand, learned, msg, scrubbed=False)   # ids, counts and times: nothing personal
    return dict(ok=True, pending=branch, before=len(before), after=len(text), cost=getattr(chat, 'cost', 0))

def compact_learned(
    learned,                # the learned folder
    target=COMPACT_TARGET,  # characters to aim the whole `AGENTS.md` at
    chat=None,              # a function from prompt to reply; None asks `editor` in one headless Ramabana turn
    editor=None,            # the model that rewrites; Ramabana's routing default when None
    exe=None,               # the Ramabana command; the one beside this Python when None
    onto=None,              # a pending id to fold in: the file it would make is compacted, and accepting the compaction accepts it too
):
    "Propose a shorter `AGENTS.md` for `learned` on a `pending/` branch: `{ok, pending, before, after, cost}`, or `{ok: False, error}`."
    try:
        with learned_lock(learned): return _compact(learned, target, chat or _editor(editor, exe), onto)
    except BatchBusy: return dict(ok=False, error='a Dhrona batch is running on the learned folder; compact it when the batch ends')

def _due_compaction(learned, at, chat):
    "`_compact` when `AGENTS.md` is past `at` of `LEARNED_MAX`, holds a paragraph Dhrona proposed, and no compaction of this HEAD was tried; else None."
    learned = Path(learned)
    if not at or not (learned/'.git').exists() or len(_show(learned, 'HEAD')) <= at*LEARNED_MAX: return None
    tried, head = learned.parent/f'.{learned.name}.compacted', _git(learned, 'rev-parse', 'HEAD')
    if (tried.exists() and tried.read_text().strip() == head) or not _git(learned, 'ls-files', 'AGENTS.md') or _pending_compaction(learned): return None
    if not any(mine for _, mine, _ in _paragraphs(learned)): return None
    tried.write_text(head)   # beside the folder, as the lock is; a rejected compaction is not proposed again until HEAD moves
    return _compact(learned, COMPACT_TARGET, chat)

def _compaction_note(c):
    "The nightly note's line for a compaction, when one was tried."
    if not c: return []
    return [(f"compaction pending {c['pending']}" if c['ok'] else f"no compaction: {c['error']}") + (f" (${c['cost']:.2f})" if c.get('cost') else '')]

In [ ]:
with TemporaryDirectory() as d:
    learned = Path(d)/'learned'; learned.mkdir()
    def commit(text, msg, who=_AS_DHRONA):
        (learned/'AGENTS.md').write_text(text); _git(learned, 'add', '-A'); _git(learned, *who, 'commit', '-qm', msg)
    head = '# Rules\n\nA rule a person wrote. Mail bob.ray@example.com.\n'
    (learned/'AGENTS.md').write_text(head); learned_repo(learned)
    tests = 'Run the tests with:\n```\nuv run pytest -q\n```'
    lessons = [f'Lesson {i}: read file sizes with ls -l, not stat. ' + 'Say it again. ' * 12 for i in range(6)] + [tests]
    text = head
    for l in lessons: text += f'\n{l}\n'; commit(text, f'{LESSON}: {l[:9]}')
    commit(text + '\nA closing rule a person wrote.\n', 'by hand', ('-c', 'user.name=t', '-c', 'user.email=t@x'))
    before = (learned/'AGENTS.md').read_text()
    paras = _paragraphs(learned)
    test_eq([mine for _, mine, _ in paras], [False, False, *[True] * 7, False])
    sent = []
    def chat(reply):
        def f(prompt): sent.append(prompt); return reply
        return f
    good = f'[kept 1]\n\n[kept 2]\n\nRead file sizes with ls -l, not stat. Ask ann.lee@example.com.\n\n{tests}\n\n[kept 3]'
    r = compact_learned(learned, chat=chat(f'```markdown\n{good}\n```'))   # one fence around the whole reply is unwrapped
    assert r['ok'], r
    out = subprocess.run(['git', '-C', str(learned), 'show', f"{r['pending']}:AGENTS.md"], capture_output=True, text=True).stdout
    assert out.startswith(head) and out.endswith('\n\nA closing rule a person wrote.\n')   # a person's paragraphs, byte for byte
    assert tests in out and '[EMAIL_1]' in out and 'ann.lee' not in out   # the fence whole, the rewrite scrubbed
    test_eq((r['before'], r['after']), (len(before), len(out)))
    assert r['after'] < COMPACT_TARGET and (learned/'AGENTS.md').read_text() == before   # HEAD waits for review
    assert 'bob.ray' not in sent[0] and '[kept 3]' in sent[0] and 'Lesson 5' in sent[0]   # the prompt is scrubbed
    subj = _git(learned, 'log', '-1', '--format=%s', r['pending'])
    test_eq(subj, f'{COMPACT} ({len(before):,} → {len(out):,} characters)')
    assert 'already pending' in compact_learned(learned, chat=chat(good))['error']
    _git(learned, *_AS_DHRONA, 'cherry-pick', r['pending'])   # what `ramabana-learned accept` does
    test_eq([mine for _, mine, _ in _paragraphs(learned)], [False, False, True, True, False])   # the compaction is Dhrona's own, so it expires as a lesson does
    _git(learned, 'branch', '-D', r['pending'])
    text = (learned/'AGENTS.md').read_text()
    commit(text + '\n' + 'Another lesson. ' * 20 + '\n', f'{LESSON}: more')
    nothing = lambda: test_eq(_git(learned, 'branch', '--list', 'pending/*'), '')
    def refused(reply, why):
        assert why in (e := compact_learned(learned, chat=chat(reply))['error']), e
        nothing()
    refused('[kept 1]\n\n[kept 2]\n\nShort.\n\n[kept 3]', 'fenced block')
    refused(f'[kept 1]\n\n[kept 2]\n\n{tests}\n\n' + 'Longer. ' * 300 + '\n\n[kept 3]', 'no shorter')
    refused(f'[kept 1]\n\nA rule a person rewrote.\n\n{tests}\n\n[kept 3]', 'a person wrote')
    refused(f'[kept 1]\n\n[kept 2]\nand more\n\n{tests}\n\n[kept 3]', 'a person wrote')
    refused(f'[kept 2]\n\n[kept 1]\n\n{tests}\n\n[kept 3]', 'a person wrote')
    refused('', 'no file')
    with learned_lock(learned): refused(good, 'batch is running')
    (learned/'AGENTS.md').write_text('an uncommitted edit\n')
    r = compact_learned(learned, chat=chat(good))   # an edit does not stop a compaction, which works from HEAD
    assert r['ok'], r
    test_eq((learned/'AGENTS.md').read_text(), 'an uncommitted edit\n')
    _git(learned, 'branch', '-D', r['pending']); _git(learned, 'checkout', '-q', '--', '.')
    plain = Path(d)/'plain'; plain.mkdir(); (plain/'AGENTS.md').write_text('Only a person wrote this.\nAnd this, right after.\n'); learned_repo(plain)
    assert 'nothing Dhrona proposed' in compact_learned(plain, chat=chat('x'))['error']

In [ ]:
with TemporaryDirectory() as d:
    d = Path(d); learned, cfg = d/'learned', d/'cfg'; learned.mkdir(); cfg.mkdir()
    (learned/'AGENTS.md').write_text('A rule a person wrote.\n'); learned_repo(learned)
    old, text = str(int(time.time()) - 40*86400), 'A rule a person wrote.\n'
    def lesson(add, when=None, msg=f'{LESSON}: x'):
        global text
        text += f'\n{add}\n'; (learned/'AGENTS.md').write_text(text)
        env = os.environ | ({'GIT_COMMITTER_DATE': when, 'GIT_AUTHOR_DATE': when} if when else {})
        subprocess.run(['git', '-C', str(learned), 'add', '-A'], check=True)
        subprocess.run(['git', '-C', str(learned), *_AS_DHRONA, 'commit', '-qm', msg], check=True, env=env)
    lesson('Read file sizes with ls -l, never stat. ' * 8, old)
    lesson('Prefer listing directories with ls before reading files. ' * 8, old)
    lesson('Search code with rg, keeping globs narrow. ' * 8)
    chat = lambda p: '[kept 1]\n\nRead file sizes with ls -l, never stat; list directories with ls before reading files.\n\nSearch code with rg, keeping globs narrow.'
    chat.cost = 0.25
    r = compact_learned(learned, chat=chat)
    test_eq((r['ok'], r['cost']), (True, 0.25))
    assert TIMES in _git(learned, 'log', '-1', '--format=%B', r['pending'])
    _git(learned, *_AS_DHRONA, 'cherry-pick', r['pending']); _git(learned, 'branch', '-D', r['pending'])   # accepted today
    (cfg/'agent-history.jsonl').write_text(json.dumps(dict(session='s', at=time.time(), activity=[])) + '\n')
    gone = expire_learned(learned, cfg)
    test_eq(len(gone), 1); assert gone[0].startswith('AGENTS.md: Read file sizes')   # merged from two 40-day-old lessons: it expires by them, not by the accept
    test_eq((learned/'AGENTS.md').read_text(), 'A rule a person wrote.\n\nSearch code with rg, keeping globs narrow.\n')
    a = d/'cand'; shutil.copytree(learned, a, ignore=_HIDDEN)
    (a/'AGENTS.md').write_text((learned/'AGENTS.md').read_text() + '\n' + 'Run tests with pytest -q before any commit. ' * 45 + '\n')
    (a/'skills'/'sizes').mkdir(parents=True); (a/'skills'/'sizes'/'SKILL.md').write_text('---\nname: sizes\ndescription: x\n---\nuse ls\n')
    big = '20261019-084007-bcbc13'   # an id `scrub` would mask as a card number
    _git(learned, 'branch', '-m', keep(a, learned, f'{LESSON}: tests'), PENDING + big)
    assert len(_show(learned, PENDING + big)) > LEARNED_MAX   # too long to accept
    assert 'nothing is pending' in compact_learned(learned, chat=chat, onto='20990101-000000-abcdef')['error']
    r = compact_learned(learned, chat=lambda p: '[kept 1]\n\nSearch code with rg, keeping globs narrow.\n\nRun tests with pytest -q before any commit.', onto=big)
    assert r['ok'] and r['before'] > LEARNED_MAX and r['after'] < COMPACT_TARGET, r
    body = _git(learned, 'log', '-1', '--format=%B', r['pending'])
    assert body.startswith(f'{COMPACT} with {big} (') and f'{INCLUDES}{big}' in body
    test_eq(_git(learned, 'ls-tree', '-r', '--name-only', r['pending']), 'AGENTS.md\nskills/sizes/SKILL.md')   # the candidate's skill comes too
    test_eq(_git(learned, 'diff', '--name-only', 'HEAD', r['pending']), 'AGENTS.md\nskills/sizes/SKILL.md')   # one commit off HEAD, so accepting it is one pick

## History health

Ramabana's history is a log nobody reads until it is 224 MB. `history_health` makes one streaming pass over it, one line at a time, and reports its size, the rows and bytes of each day, the largest row, and the groups of failing tool calls. A group is a tool plus the first 60 characters of an `ERROR` result, or of a `TypeError` about a missing positional argument, counted over the last `days` and over the `days` before. A `defect` is a tool that crashed, a `habit` is the model or the guard disagreeing about how to use a tool, and `noise` is a shell command that exited non-zero.

In [ ]:
#| export
LOG_MAX_MB = 200      #: a history log past this many megabytes earns a line in the night's note
LOG_GROWTH_MB = 20    #: so does one that grew faster than this many megabytes a day over the last 7 days
DEFECT_MAX = 20       #: a defect or habit group seen more often than this in `days` days is reported; a defect that doubled is too
ROW_MAX_MB = 5        #: a single history row past this many megabytes earns a line
_MB = 10**6

_MISSING = re.compile(r'TypeError\b.*\bmissing \d+ required positional arguments?', re.S)
_GUARD = re.compile(r'cuts the output a failure is in|merges stderr into stdout|goes through `\w+` here')   # what `run_shell`'s guard says when it refuses
_STATE = re.compile(r'^state:\s*\w+', re.M)
_SUBAGENT = 'delegate_search'   #: the tool whose result must end in a `state:` line

def _group_kind(tool, text):
    "`defect` for a tool that crashed, `noise` for a shell command that exited non-zero, else `habit`."
    if 'TypeError' in text: return 'defect'
    return 'noise' if tool == 'run_shell' and re.match(r'ERROR: exit \d+', text) else 'habit'

def history_health(
    cfg='~/.config/ramabana',   # Ramabana's config folder, whose `agent-history.jsonl` is read
    days=14,                    # the window; the `days` before it give each group's `previous` count
    now=None,                   # epoch seconds; the present when None
    top=20,                     # groups to return, most frequent first
    fixed=None,                 # `load_fixed` rows: a group's occurrences at or before its fix time are not counted
):
    "One streaming pass over `cfg`'s history: log size, rows and bytes per day, the largest row, failing-call groups, guard refusals, repeated calls and sub-agent results without a `state:` line."
    p = Path(cfg).expanduser()/'agent-history.jsonl'
    now = time.time() if now is None else now
    mid, lo = now - days*86400, now - 2*days*86400
    per_day, largest, groups, tally = {}, dict(bytes=0, session=None, at=None), {}, Counter()
    with open(p, 'rb') as f:
        for line in f:
            try: t = json.loads(line)
            except ValueError: continue
            if not isinstance(t, dict): continue
            at = _epoch(t.get('at'))
            try: day = datetime.fromtimestamp(at, timezone.utc).strftime('%Y-%m-%d') if at else 'unknown'
            except (OverflowError, OSError, ValueError): day = 'unknown'
            v = per_day.setdefault(day, dict(rows=0, bytes=0)); v['rows'] += 1; v['bytes'] += len(line)
            if len(line) > largest['bytes']: largest = dict(bytes=len(line), session=t.get('session'), at=at)
            win = 'now' if mid < at <= now else 'previous' if lo < at <= mid else None
            if win is None: continue
            last = None
            for a in t.get('activity') or []:
                if not isinstance(a, dict): continue
                tool, text = a.get('tool') or '', str(a.get('detail') or '')
                same = (tool, json.dumps(tool_args(a), sort_keys=True, default=str))
                if win == 'now':
                    tally['repeats'] += same == last
                    tally['guard_refusals'] += tool == 'run_shell' and bool(_GUARD.search(text))
                    tally['no_state'] += tool == _SUBAGENT and not _STATE.search(text)
                last = same
                if not (text.startswith('ERROR') or _MISSING.search(text)): continue
                fix = (fixed or {}).get(_slug(dict(tool=tool, key=text[:60])))
                if fix and at <= _epoch(fix['at']): continue
                g = groups.setdefault((tool, text[:60]), dict(tool=tool, key=text[:60], kind=_group_kind(tool, text), count=0, previous=0, examples=[], raw=text[:1000], fixed=fix))
                if win == 'previous': g['previous'] += 1; continue
                g['count'] += 1
                if g['count'] == 1: g['raw'] = text[:1000]
                if len(g['examples']) < 3 and t.get('session') not in {e['session'] for e in g['examples']}:
                    g['examples'].append(dict(session=t.get('session'), at=at))
    return dict(log_bytes=p.stat().st_size, per_day=per_day, largest=largest, days=days, now=now,
                groups=sorted(groups.values(), key=lambda g: (-g['count'], -g['previous']))[:top],
                guard_refusals=tally['guard_refusals'], repeats=tally['repeats'], no_state=tally['no_state'])

A night's note gets a line for each threshold the report crosses: a log over `LOG_MAX_MB` or growing over `LOG_GROWTH_MB` a day for a week, a row over `ROW_MAX_MB`, and a defect group seen more than `DEFECT_MAX` times or twice as often as in the window before. Each such defect also gets an issue draft in the night's folder, with its count and trend, three example sessions, one raw error and the tool. A habit is never an issue: past `DEFECT_MAX` it becomes a candidate lesson, which a person reviews before anything is learned.

A defect a person has fixed goes in the fixed-defects record, `FIXED` (`~/.config/dhrona/fixed.json`, moved by `XDG_CONFIG_HOME`), keyed by the slug that names its issue draft. Each entry holds `at`, when the fix was made, a `note` such as a commit or a version, and an optional `check`. `record_fix` writes an entry, and `dhrona-fixed SLUG --note NOTE --check CHECK` does the same from a shell. `history_health` counts only the occurrences after `at`. A defect with any occurrence after `at` is reported as `recurred after fix`, whatever its count, and its issue draft says so. A defect with none is not reported and gets no draft.

A `check` is a shell command or a `module:callable` path. A shell command passes on exit 0. A callable passes when it returns without raising and does not return `False`. A check that runs longer than 60 seconds fails. Before each night's batch, `nightly` runs every recorded check once (`check_lines`) and puts each failure in the note as `fix check failed for SLUG`.

In [ ]:
#| export
def health_findings(h):
    "`(lines, defects)` for a `history_health` report: a line for each threshold it crosses, and the defect groups that crossed one."
    lines = []
    if h['log_bytes'] > LOG_MAX_MB*_MB: lines.append(f"history log is {h['log_bytes']/_MB:.0f} MB, past {LOG_MAX_MB} MB")
    since = datetime.fromtimestamp(h['now'] - 7*86400, timezone.utc).strftime('%Y-%m-%d')
    rate = sum(v['bytes'] for k, v in h['per_day'].items() if k != 'unknown' and k >= since) / 7
    if rate > LOG_GROWTH_MB*_MB: lines.append(f'history log grew {rate/_MB:.0f} MB a day over the last 7 days, past {LOG_GROWTH_MB} MB')
    big = h['largest']
    if big['bytes'] > ROW_MAX_MB*_MB: lines.append(f"largest history row is {big['bytes']/_MB:.1f} MB (session {big['session']}), past {ROW_MAX_MB} MB")
    defects = [g for g in h['groups'] if g['kind'] == 'defect' and ((g.get('fixed') and g['count']) or g['count'] > DEFECT_MAX or (g['previous'] and g['count'] >= 2*g['previous']))]
    for g in defects:
        if (fix := g.get('fixed')) and g['count']: lines.append(f"defect {g['tool']} recurred after fix ({fix.get('note') or 'no note'}, fixed {fix['at']}): {g['count']} since: {g['key']}")
        else: lines.append(f"defect {g['tool']}: {g['count']} in {h['days']} days, {g['previous']} in the {h['days']} before: {g['key']}")
    return lines, defects

def _slug(g): return re.sub(r'[^a-z0-9]+', '-', f"{g['tool']} {g['key']}".lower()).strip('-')[:60]

def _iso(at): return datetime.fromtimestamp(at, timezone.utc).isoformat(timespec='seconds') if at else 'unknown'

def issue_drafts(defects, work, days=14):
    "Write `issues/<slug>.md` in `work` for each defect group, with its count and trend, example sessions, one raw error and the tool; returns the paths."
    out = []
    for g in defects:
        trend = 'new' if not g['previous'] else f"{g['count']/g['previous']:.1f}x the {g['previous']} before"
        sessions = '\n'.join(f"- `{e['session']}` at {_iso(e['at'])}" for e in g['examples'])
        fix = g.get('fixed')
        again = f"- recurred after fix: {fix.get('note') or 'no note'}, fixed {fix['at']}\n" if fix and g['count'] else ''
        text = (f"# {g['tool']} fails: {g['key']}\n\n- tool: `{g['tool']}`\n- slug: `{_slug(g)}`; after a fix, run `dhrona-fixed {_slug(g)}`\n{again}- count: {g['count']} in the last {days} days, {g['previous']} in the {days} before ({trend})\n\n"
                f"## Sessions\n\n{sessions}\n\n## Raw error\n\n```\n{scrub(g['raw'])}\n```\n")
        (p := Path(work)/'issues'/f'{_slug(g)}.md').parent.mkdir(parents=True, exist_ok=True)
        p.write_text(text); out.append(p)
    return out

FIXED = xdg_config_home()/'dhrona'/'fixed.json'   #: the defects a person has fixed, by `_slug`: when, a note, and an optional quick check

def load_fixed(path=FIXED):
    "The fixed-defects record at `path`, slug to `{at, note, check}`; empty when there is none."
    return _json(Path(path).expanduser()) or {}

def record_fix(
    slug,         # the defect's `_slug`, the name of its file in a night's `issues/` folder
    note='',      # what fixed it, such as a commit or version
    check='',     # a shell command, or a `module:callable` path, that passes once the defect is gone
    at=None,      # when it was fixed, an ISO date or epoch seconds; now when None
    path=FIXED,   # the record
):
    "Record that defect `slug` was fixed, so later nights count only what happens after `at`; returns its entry."
    when = time.time() if at is None else _epoch(at)
    if not when: raise ValueError(f'cannot read the fix time {at!r}')
    path = Path(path).expanduser()
    path.parent.mkdir(parents=True, exist_ok=True)
    fixed = load_fixed(path)
    fixed[slug] = entry = dict(at=_iso(when), note=note, check=check)
    path.write_text(json.dumps(fixed, indent=1))
    return entry

_CALLABLE = re.compile(r'[\w.]+:[\w.]+')

def run_check(check, timeout=60):
    "None when the quick `check` passes, else why not: a shell command passes on exit 0, a `module:callable` path on returning without raising or returning False."
    try:
        if _CALLABLE.fullmatch(check):
            mod, _, name = check.partition(':')
            f = importlib.import_module(mod)
            for part in name.split('.'): f = getattr(f, part)
            return 'returned False' if f() is False else None
        p = subprocess.run(check, shell=True, stdin=subprocess.DEVNULL, capture_output=True, text=True, timeout=timeout)
    except subprocess.TimeoutExpired: return f'still running after {timeout}s'
    except Exception as e: return f'{type(e).__name__}: {e}'   # a callable may raise anything
    return f'exit {p.returncode}: {(p.stderr or p.stdout).strip()[:200]}' if p.returncode else None

def check_lines(fixed):
    "The night's note lines for the quick checks in `fixed` that fail; each check runs once."
    return [f"fix check failed for {slug} ({e.get('note') or 'no note'}): {why}" for slug, e in fixed.items() if e.get('check') and (why := run_check(e['check']))]

@call_parse
def fixed_cli(
    slug:str,          # the defect's slug, the name of its file in a night's `issues/` folder
    note:str='',       # what fixed it, such as a commit or version
    check:str='',      # a shell command, or a `module:callable` path, that passes once the defect is gone
    at:str=None,       # when it was fixed, an ISO date or epoch seconds; now when omitted
    path:str=str(FIXED), # the fixed-defects record
):
    "Record that a defect Dhrona reports was fixed: later nights count only what happens after, and report a recurrence."
    print(f"{slug} fixed at {record_fix(slug, note, check, at, path)['at']}")

def lesson_candidates(h, work):
    "Write `lessons-candidates.md` in `work`, as `lessons_md` renders lessons, for each habit group past `DEFECT_MAX`; returns its path, or None when there is none."
    rows = [dict(lesson=scrub(f"`{g['tool']}` keeps failing with `{g['key']}` ({g['count']} times in {h['days']} days)"), component='rule')
            for g in h['groups'] if g['kind'] == 'habit' and g['count'] > DEFECT_MAX]
    if not rows: return None
    (p := Path(work)/'lessons-candidates.md').parent.mkdir(parents=True, exist_ok=True)
    p.write_text(lessons_md(rows))
    return p

In [ ]:
with TemporaryDirectory() as d:
    d = Path(d); now = 1_700_000_000.
    ex = [dict(session=f's{n}', at=now - n*3600) for n in (1, 2, 3, 4)]
    def grp(tool, key, kind, count, previous): return dict(tool=tool, key=key, kind=kind, count=count, previous=previous, examples=ex[:3], raw=f'{key} in full')
    h = dict(now=now, days=14, log_bytes=250*10**6, largest=dict(bytes=6*10**6, session='big', at=now),
             per_day={f'2023-11-{n:02d}': dict(rows=9, bytes=30*10**6) for n in range(7, 15)},
             groups=[grp('grep', 'TypeError: grep() missing 1 required positional argument', 'defect', 21, 0), grp('ls', 'TypeError: ls() missing 1 required positional argument', 'defect', 4, 2),
                     grp('stat', 'TypeError: stat() missing 1 required positional argument', 'defect', 5, 0), grp('view_file', 'ERROR: Unsafe path', 'habit', 50, 40)])
    lines, defects = health_findings(h)
    test_eq([g['tool'] for g in defects], ['grep', 'ls'])   # over the limit, and doubled; five a fortnight is neither
    assert len(lines) == 5 and 'MB' in lines[0] and 'a day' in lines[1] and 'big' in lines[2] and lines[3].startswith('defect grep')
    quiet = h | dict(log_bytes=10, largest=dict(bytes=1, session='x', at=now), per_day={}, groups=h['groups'][2:3])
    test_eq(health_findings(quiet), ([], []))
    paths = issue_drafts(defects, d/'night')
    test_eq(sorted(p.parent.name for p in paths), ['issues', 'issues'])
    text = paths[0].read_text()
    assert 'grep' in text and '21' in text and 's1' in text and 's3' in text and 's4' not in text and 'missing 1 required positional argument in full' in text
    assert '2023-11-14' in text   # an example's timestamp
    cand = lesson_candidates(h, d/'night')
    assert 'view_file' in cand.read_text() and 'TypeError' not in cand.read_text()
text

In [ ]:
with TemporaryDirectory() as d:
    d = Path(d); now, day = 1_700_000_000., 86400
    def row(ago, sid, *acts): return dict(session=sid, at=now - ago*day, activity=list(acts))
    def call(tool, detail, ok=False, **args): return dict(tool=tool, ok=ok, args=args, detail=detail)
    miss = "TypeError: code_tools.<locals>.grep() missing 1 required positional argument: 'pattern'"
    rows = [row(1, 's1', call('grep', miss)), row(2, 's2', call('grep', miss)), row(3, 's3', call('grep', miss)), row(20, 's0', call('grep', miss)),
            row(1, 's4', call('run_shell', 'ERROR: exit 1', command='a'), call('run_shell', 'ERROR: exit 2', command='b')),
            row(1, 's5', call('run_shell', '`| tail` cuts the output a failure is in. Run the command bare', command='ls | tail')),
            row(1, 's6', call('ls', 'a', True, path='.'), call('ls', 'a', True, path='.'), call('ls', 'a', True, path='/etc')),
            row(1, 's7', call('delegate_search', 'found it\nstate: completed', True, q='a'), call('delegate_search', 'found nothing', True, q='b')),
            row(1, 's8', call('view_file', 'ERROR: Unsafe path /etc/passwd is outside the open folders', path='/etc/passwd')),
            row(1, 'big', call('ls', 'x'*3000, True))]
    (d/'agent-history.jsonl').write_text('\n'.join(map(json.dumps, rows)) + '\nnot json\n')
    h = history_health(d, days=14, now=now)
    g = {(x['tool'], x['kind']): x for x in h['groups']}
    test_eq((g['grep', 'defect']['count'], g['grep', 'defect']['previous']), (3, 1))
    test_eq(sorted(e['session'] for e in g['grep', 'defect']['examples']), ['s1', 's2', 's3'])
    test_eq(g['grep', 'defect']['raw'], miss)
    test_eq([x['kind'] for x in h['groups'] if x['tool'] == 'run_shell'], ['noise', 'noise'])
    test_eq(g['view_file', 'habit']['count'], 1)
    test_eq((h['guard_refusals'], h['repeats'], h['no_state']), (1, 1, 1))
    test_eq((h['log_bytes'], h['largest']['session']), ((d/'agent-history.jsonl').stat().st_size, 'big'))
    assert h['largest']['bytes'] > 3000 and sum(v['rows'] for v in h['per_day'].values()) == len(rows)
    test_fail(lambda: history_health(d/'nowhere'), exc=FileNotFoundError)
    # examples name distinct sessions, so one session failing three times does not fill every slot
    (d/'agent-history.jsonl').write_text('\n'.join(map(json.dumps, [row(1, 'sA', call('grep', miss), call('grep', miss), call('grep', miss)), row(1, 'sB', call('grep', miss))])) + '\n')
    test_eq([e['session'] for e in history_health(d, days=14, now=now)['groups'][0]['examples']], ['sA', 'sB'])
[(x['tool'], x['kind'], x['count'], x['previous']) for x in h['groups'][:3]]

A fix is recorded once and counted from its time. In the history below, `grep` fails three times before the fix and once after it. Without the record the group counts four. With it the group counts the one failure after `at`, and `health_findings` reports it as `recurred after fix`, although one is far below `DEFECT_MAX`. A fix made after the last failure leaves no group.

In [ ]:
with TemporaryDirectory() as d:
    d = Path(d)
    hist = [row(ago, f's{ago}', call('grep', miss)) for ago in (5, 4, 3, 1)]   # days ago; the fix is 2 days ago
    (d/'agent-history.jsonl').write_text('\n'.join(map(json.dumps, hist)) + '\n')
    slug = _slug(dict(tool='grep', key=miss[:60]))
    test_eq(history_health(d, now=now)['groups'][0]['count'], 4)
    path = d/'fixed.json'
    entry = record_fix(slug, note='0.1.19', at=_iso(now - 2*day), path=path)
    test_eq(load_fixed(path), {slug: entry})
    h = history_health(d, now=now, fixed=load_fixed(path))
    g, = h['groups']
    test_eq((g['count'], g['fixed']['note']), (1, '0.1.19'))   # the three failures before the fix are not counted
    lines, defects = health_findings(h)
    test_eq(len(defects), 1)
    assert 'recurred after fix (0.1.19' in lines[0], lines
    assert 'recurred after fix' in issue_drafts(defects, d/'night')[0].read_text()
    test_eq(history_health(d, now=now, fixed={slug: dict(at=_iso(now - day/2))})['groups'], [])   # a fix after the last failure leaves nothing to report
    test_fail(lambda: record_fix(slug, at='soon', path=path), contains='fix time')
    test_eq(check_lines({'ok': dict(check='true'), 'py': dict(check='platform:system'), 'none': dict(note='no check')}), [])
    bad = check_lines({'b': dict(check='exit 3', note='v1'), 'c': dict(check='json:nothing')})
    assert bad[0].startswith('fix check failed for b (v1): exit 3') and 'AttributeError' in bad[1], bad
bad

## Every night

`nightly` is the function a scheduled beat calls. `ramabana-tick` finds it through the `ramabana.ticks` entry point named `dhrona`, and calls it when the pobblebonk schedule named `dhrona` fires, passing the schedule's payload as `fire`. The payload must hold a positive `budget` in dollars, or `nightly` raises `ValueError` before any round runs. A `timeout` that is not a positive number, given as a number or as text, raises the same way. The other keys are optional:

| key | meaning | default |
|---|---|---|
| `n` | rounds | 3 |
| `k` | base runs per round | 2 |
| `model` | the model the replays use | each session's own |
| `editor` | the model that edits and judges | Ramabana's default |
| `models` | a list of model ids whose sessions are kept | all |
| `max_calls` | the most calls a replayed turn may have made | 40 |
| `cfg` | the Ramabana config folder | `~/.config/ramabana` |
| `learned` | the learned folder | `<cfg>/learned`, the folder Ramabana loads |
| `work` | where batch folders go | `NIGHTLY`, `~/.cache/dhrona/nightly` |
| `expire_days` | days of history the lessons come from, and that a proposed skill or rule may go unused before it expires | 30 |
| `compact_at` | the fraction of `LEARNED_MAX` past which `AGENTS.md` gets a compaction proposed | 0.8 |
| `timeout` | seconds before a replay, base or candidate, is abandoned; its batch row then carries `the run in ... was still going after Ns` | 1800 |
| `exe` | the Ramabana command | see below |
| `fixed` | the fixed-defects record (see History health) | `FIXED`, `~/.config/dhrona/fixed.json` |

Each night writes a new batch folder under `work`, named by the time it started. It skips every session that a batch under `work` has already started a round on (`tried_sessions`), including rounds that failed, and when no session is left it creates nothing. A night with no session left still expires what went unused. `nightly` returns the note the beat leaves for the next session: how many rounds ran, the pending branches waiting for review, what expired, each recorded fix check that failed, a compaction it proposed, and the dollars spent against the budget.

After expiry and before the rounds, a night proposes a compaction when `AGENTS.md` is over `compact_at` of `LEARNED_MAX`, holds a paragraph Dhrona proposed, and has no compaction pending. The note says `compaction pending <branch>`, or why none was proposed. A night tries one compaction per HEAD: the HEAD it tried is kept in a file beside the learned folder, so a compaction a person rejected is not proposed again until HEAD moves. The editor's turn counts against `budget`: its cost is in the night's total, in the note's compaction line and in `compaction.json`, and a compaction that spends the budget leaves no round to run.

The note names each failed round with the first line of its error (`night_error`). Where Ramabana's error suggests `/model NAME`, which only an interactive session can act on, the note names the schedule key to set instead: `model` when the replay's own model could not run, `editor` otherwise. An error saying Claude Code is not logged in, as happens when a job started by launchd cannot read the login from the keychain, gets a line saying to run `claude /login` or to put `ANTHROPIC_API_KEY` in Ramabana's `beat.env`, which bills the API instead of the subscription.

A night where every round failed and nothing was spent could not run at all: a missing tool, a model that does not start, a lost login. It writes the note into a `night-failed` file in its batch folder and raises `NightFailed` with the note as its message, so pobblebonk retries the fire and, after the last try, leaves the note as `gave up after N tries`. `tried_sessions` does not count a session from such a night until it has sat in `FAILED_TRIES` (2) of them, so the retry replays the same sessions and a session that never runs is eventually passed over. A night with no session left, or with one round that ran, returns its note and is not retried.

If a batch is already running on the learned folder, the night returns a note saying so and runs nothing. pobblebonk 0.0.4 and later renew a running fire's claim, so this only happens when a second beat starts while one is still running. Any other failure raises, and pobblebonk retries a fire that raised, up to the schedule's `retries`. Add the schedule with `retries=2` and `retry_after='1h'`, so a night that could not run gets one more try an hour later (pobblebonk 0.0.5; an older pobblebonk retries after 30 seconds instead):

```python
from ramabana.monitor import pob
pob().add('dhrona', cron='0 3 * * *', retries=2, retry_after='1h', budget=20, n=3)
```

An existing schedule takes the same change with `pob().update('dhrona', retries=2, retry_after='1h')`. Each try runs replays under `timeout`, and the beat that runs them stops itself after `ramabana-tick --limit`, so a night cannot hold the beat forever.

Without `exe`, a night runs the `ramabana` installed beside the Python that runs the beat, or else the one found on `PATH`. This matters because a beat started by cron or launchd has a short `PATH`.

In [ ]:
#| export
NIGHTLY = '~/.cache/dhrona/nightly'   #: where scheduled batches write, one folder a night

FAILED = 'night-failed'   #: the marker a night where nothing could run leaves in its batch folder
FAILED_TRIES = 2          #: such nights a session sits in before it counts as tried

def tried_sessions(root):
    "Session ids that the batches under `root` have already started a round on; a night where nothing could run counts after `FAILED_TRIES`."
    dirs = [p for p in Path(root).expanduser().glob('*/*') if p.is_dir()]
    lost = Counter(p.name for p in dirs if (p.parent/FAILED).exists())
    return {p.name for p in dirs if not (p.parent/FAILED).exists() or lost[p.name] >= FAILED_TRIES}

class NightFailed(RuntimeError): "Every round of a night failed and nothing was spent, so the beat retries the fire."

_PICK = re.compile(r"(\S+) runs here; choose it with `/model \S+` or the model picker")
_TURN = re.compile(r"turn job: (\S+) cannot run here")
_LOGIN = re.compile(r'not logged in|invalid api key|please run /login|keychain', re.I)

def night_error(row):
    "The first line of a failed round's error, naming the schedule key a night reads where Ramabana says `/model`, and what to do about a lost Claude login."
    err = next(iter(str(row.get('error') or '').strip().splitlines()), '')[:400]
    key = 'model' if (m := _TURN.search(err)) and m[1] == row.get('model') else 'editor'
    err = _PICK.sub(lambda m: f"{m[1]} runs here; set the schedule's `{key}`: pob().update('dhrona', {key}='{m[1]}')", err)
    if _LOGIN.search(err): err += ('. Claude Code could not use its login from the beat: run `claude /login` in a terminal, or put ANTHROPIC_API_KEY'
                                   ' in ~/.config/ramabana/beat.env, which bills the API instead of the subscription')
    return err

def _health_lines(cfg, work=None, fixed=None):
    "The beat lines for what `history_health` finds over its thresholds, counting a defect in `fixed` only after its fix; with a `work` folder, also the issue drafts and lesson candidates it writes there."
    try: h = history_health(cfg, fixed=fixed)
    except OSError as e: return [f'history health unavailable: {e}']
    lines, defects = health_findings(h)
    if work is not None and defects: issue_drafts(defects, work, h['days']); lines.append(f"issue drafts in {Path(work)/'issues'}")
    if work is not None and (p := lesson_candidates(h, work)): lines.append(f'lesson candidates in {p}')
    return lines

def nightly(fire):
    "One scheduled batch over sessions no earlier night tried, within `fire.budget` dollars; returns the note the beat leaves."
    if not (fire.get('budget') or 0) > 0: raise ValueError('the dhrona schedule needs a positive `budget` in dollars')
    try: timeout = 1800.0 if fire.get('timeout') is None else float(fire['timeout'])   # a payload may hold the number as text
    except (TypeError, ValueError): timeout = 0
    if not timeout > 0: raise ValueError(f"the dhrona schedule's `timeout` must be a positive number of seconds, not {fire['timeout']!r}")
    cfg, root = fire.get('cfg', '~/.config/ramabana'), Path(fire.get('work', NIGHTLY)).expanduser()
    fixed = load_fixed(fire.get('fixed', FIXED))
    broken = check_lines(fixed)
    tried = tried_sessions(root)
    left = [s for s in trainable_sessions(cfg, fire.get('max_calls', 40)) if s['session'] not in tried]
    learned, days = Path(fire.get('learned', Path(cfg)/'learned')).expanduser(), fire.get('expire_days', 30)
    exe, at = fire.get('exe') or _beside_python(), fire.get('compact_at', 0.8)
    if not left:
        try:
            with learned_lock(learned): gone, c = expire_learned(learned, cfg, days), _due_compaction(learned, at, _editor(fire.get('editor'), exe))
        except BatchBusy as e: return f'a batch started earlier is still running: {e}'
        return '\n'.join(['no session left that an earlier night has not tried', *(f'expired {g}' for g in gone), *_compaction_note(c),
                          *broken, *_health_lines(cfg, fixed=fixed), *([REVIEW] if c and c['ok'] else [])])
    work = root/datetime.now().strftime('%Y-%m-%d-%H%M%S')
    try: rows = improve_batch(work, learned, left, n=fire.get('n', 3), k=fire.get('k', 2),
                              model=fire.get('model'), editor=fire.get('editor'), models=fire.get('models'), exe=exe,
                              budget=fire.budget, cfg=cfg, days=days, compact_at=at, timeout=timeout)
    except BatchBusy as e: return f'a batch started earlier is still running: {e}'   # the beat took back a fire past its lease; raising would retry it
    except (RuntimeError, ValueError, OSError, subprocess.SubprocessError) as e:   # a refusal: `work` holds its marker, and a retry would meet the same one
        return '\n'.join([f'night refused: {type(e).__name__}: {e}; see {work}', *broken, *_health_lines(cfg, work, fixed)])
    kept, failed = [r for r in rows if r.get('adopted')], [r for r in rows if r.get('error')]
    gone, c = _json(work/'expired.json') or [], _json(work/'compaction.json')
    spent = sum(r['cost'] for r in rows) + ((c or {}).get('cost') or 0)
    head = f"{len(rows)} rounds, {len(kept)} pending review, {len(failed)} failed, ${spent:.2f} of ${fire.budget}; see {work}"
    note = '\n'.join([head, *(f"pending {r['pending']}: {r['prompt'][:60]}" for r in kept), *(f"failed {r['session']}: {night_error(r)}" for r in failed),
                       *(f'expired {g}' for g in gone), *_compaction_note(c), *broken, *_health_lines(cfg, work, fixed),
                       *([REVIEW] if kept or (c and c['ok']) else [])])
    if rows and len(failed) == len(rows) and not spent:
        (work/FAILED).write_text(note)
        raise NightFailed(note)
    return note

In [ ]:
import time
from fastcore.basics import AttrDict

In [ ]:
with TemporaryDirectory() as d:
    d = Path(d)
    exe, repo, cfg = d/'paid-ramabana', d/'repo', d/'cfg'
    exe.write_text(FAKE.replace("usage={'cost': 0}", "usage={'cost': 1}")); exe.chmod(0o755); repo.mkdir(); cfg.mkdir()
    def git(*a): return subprocess.run(['git', '-C', str(repo), *a], check=True, capture_output=True, text=True).stdout.strip()
    git('init', '-q'); (repo/'big.py').write_text('x = 1\n'); git('add', '.')
    git('-c', 'user.name=t', '-c', 'user.email=t@example.com', 'commit', '-qm', 'one')
    ls = dict(tool='ls', ok=True, args={'path': str(repo)})
    past = dict(session='s1', model='m1', at=time.time() + 60, reply='big.py 50',
                prompt='<user-request>\nWhat are the sizes of the big files in this repository?\n</user-request>',
                activity=[ls, dict(tool='run_shell', ok=False, args={'command': 'stat big.py'}), ls])
    (cfg/'agent-history.jsonl').write_text(json.dumps(past) + '\n')
    fire = AttrDict(budget=5, cfg=str(cfg), work=str(d/'nights'), exe=str(exe))
    note = nightly(fire)
    print(note)
    assert note.startswith('1 rounds, 1 pending review, 0 failed, $5.00 of $5') and 'pending pending/' in note and note.endswith(REVIEW)
    assert 'ramabana-learned list' not in note, 'the CLI takes `--cmd list`, so a bare `list` fails'
    assert not sorted(_files(cfg/'learned')) and 'pending/' in _git(cfg/'learned', 'branch', '--list', 'pending/*')   # waiting for review
    test_eq(tried_sessions(d/'nights'), {'s1'})
    test_eq(nightly(fire), 'no session left that an earlier night has not tried')
    test_eq(len(list((d/'nights').iterdir())), 1)   # the second night created no folder
    test_fail(lambda: nightly(AttrDict(cfg=str(cfg))), contains='budget')
    for bad in (0, -5, 'soon'): test_fail(lambda: nightly(AttrDict(fire, timeout=bad)), contains='timeout')
    test_eq(nightly(AttrDict(fire, timeout='900')), 'no session left that an earlier night has not tried')   # a payload may hold the number as text
    fixed = d/'fixed.json'
    record_fix('grep-missing', note='0.1.19', check='exit 3', path=fixed); record_fix('ls-missing', check='true', path=fixed)
    note = nightly(AttrDict(fire, fixed=str(fixed)))
    assert 'fix check failed for grep-missing (0.1.19): exit 3' in note and 'ls-missing' not in note, note   # a failing quick check is in the night's note, a passing one is not

In [ ]:
with TemporaryDirectory() as d:
    d = Path(d)
    exe, repo, cfg = d/'broken-ramabana', d/'repo', d/'cfg'
    exe.write_text('#!/bin/sh\nexit 1\n'); exe.chmod(0o755); repo.mkdir(); cfg.mkdir()   # starts no turn: a night where nothing runs
    def git(*a): return subprocess.run(['git', '-C', str(repo), *a], check=True, capture_output=True, text=True).stdout.strip()
    git('init', '-q'); (repo/'big.py').write_text('x = 1\n'); git('add', '.')
    git('-c', 'user.name=t', '-c', 'user.email=t@example.com', 'commit', '-qm', 'one')
    ls = dict(tool='ls', ok=True, args={'path': str(repo)})
    past = dict(session='s1', model='m1', at=time.time() + 60, reply='big.py 50',
                prompt='<user-request>\nWhat are the sizes of the big files in this repository?\n</user-request>',
                activity=[ls, dict(tool='run_shell', ok=False, args={'command': 'stat big.py'}), ls])
    (cfg/'agent-history.jsonl').write_text(json.dumps(past) + '\n')
    fire = AttrDict(budget=5, cfg=str(cfg), work=str(d/'nights'), exe=str(exe))
    test_fail(lambda: nightly(fire), exc=NightFailed, contains='failed s1: RuntimeError: no turn was recorded')
    test_eq(len(list((d/'nights').glob(f'*/{FAILED}'))), 1)
    test_eq(tried_sessions(d/'nights'), set())   # the retry replays the same session
    time.sleep(1.1)   # batch folders are named to the second
    test_fail(lambda: nightly(fire), exc=NightFailed)
    test_eq(tried_sessions(d/'nights'), {'s1'})   # passed over after FAILED_TRIES such nights
    test_eq(nightly(fire), 'no session left that an earlier night has not tried')

A night that refuses to start is as visible as one whose rounds all failed. The batch folder and its `night-failed` marker exist before any check that can raise, `summary.json` holds the error, and `nightly` returns a note naming the refusal. A refusal is not retried by raising, because a retry would meet the same refusal. Uncommitted files in the learned folder are not a refusal.

In [ ]:
with TemporaryDirectory() as d:
    d = Path(d)
    exe, repo, cfg = d/'fake-ramabana', d/'repo', d/'cfg'
    exe.write_text(FAKE.replace("elif (cfg/'learned'/'AGENTS.md').exists():", "elif 'sizes' in (cfg/'learned'/'AGENTS.md').read_text():")); exe.chmod(0o755); repo.mkdir(); cfg.mkdir()   # a run is cheap only under an AGENTS.md that says `sizes`
    def git(*a): return subprocess.run(['git', '-C', str(repo), *a], check=True, capture_output=True, text=True).stdout.strip()
    git('init', '-q'); (repo/'big.py').write_text('x = 1\n'); git('add', '.')
    git('-c', 'user.name=t', '-c', 'user.email=t@example.com', 'commit', '-qm', 'one')
    ls = dict(tool='ls', ok=True, args={'path': str(repo)})
    past = dict(session='s1', model='m1', at=time.time() + 60, reply='big.py 50',
                prompt='<user-request>\nWhat are the sizes of the big files in this repository?\n</user-request>',
                activity=[ls, dict(tool='run_shell', ok=False, args={'command': 'stat big.py'}), ls])
    miss = "TypeError: code_tools.<locals>.grep() missing 1 required positional argument: 'pattern'"
    bugs = [dict(session=f'b{n}', at=time.time() - 3600, activity=[dict(tool='grep', ok=False, args={}, detail=miss)]) for n in range(21)]
    (cfg/'agent-history.jsonl').write_text('\n'.join(map(json.dumps, [past, *bugs])) + '\n')
    learned, dated = cfg/'learned', f'dhrona-{datetime.now():%Y-%m-%d}.md'
    learned.mkdir(); (learned/'AGENTS.md').write_text('Use ls.\n'); learned_repo(learned)
    edit = 'Read file sizes with ls, as I edited it.\n'
    (learned/'AGENTS.md').write_text(edit)   # a person's uncommitted edit
    fire = AttrDict(budget=5, cfg=str(cfg), work=str(d/'nights'), exe=str(exe))
    note = nightly(fire)
    folder, = (d/'nights').iterdir()
    assert note.startswith('1 rounds, 1 pending review, 0 failed') and 'pending pending/' in note and not (folder/FAILED).exists()
    test_eq(json.loads((folder/'summary.json').read_text())[0]['bases'], [[0, 1, 3], [0, 1, 3]])   # the replays began from HEAD, which lacks the edit
    test_eq((learned/'AGENTS.md').read_text(), edit)
    test_eq((_git(learned, 'diff', '--name-only'), _git(learned, 'diff', '--cached', '--name-only')), ('AGENTS.md', ''))   # still modified, and not staged
    pending = json.loads((folder/'summary.json').read_text())[0]['pending']
    test_eq((_git(learned, 'diff', '--name-only', 'HEAD', pending), _git(learned, 'log', '-1', '--format=%an', pending)), (dated, 'Dhrona'))   # the branch adds the day's file alone
    assert 'defect grep: 21 in 14 days' in note and str(folder/'issues') in note   # the night names the defect and where its draft is
    test_eq(len(list((folder/'issues').glob('*.md'))), 1)
note

In [ ]:
gone = "turn job: cloud-big cannot run here: claude runtime is unavailable (`claude` is not on PATH); install Claude Code. local-small runs here; choose it with `/model local-small` or the model picker. `ramabana setup` lists the providers"
test_eq(night_error(dict(model='local-tiny', error=gone)).split('. ')[-2], "local-small runs here; set the schedule's `editor`: pob().update('dhrona', editor='local-small')")
assert "set the schedule's `model`" in night_error(dict(model='cloud-big', error=gone))   # the replay's own model could not run
assert '/model' not in night_error(dict(model='cloud-big', error=gone))
assert 'claude /login' in night_error(dict(error='RuntimeError: Not logged in · Please run /login\nmore'))
test_eq(night_error(dict(error='RuntimeError: no turn was recorded\nsecond line')), 'RuntimeError: no turn was recorded')
test_eq(night_error(dict(error=None)), '')

In [ ]:
with TemporaryDirectory() as d:
    d = Path(d)
    exe, repo, cfg = d/'fake-ramabana', d/'repo', d/'cfg'
    exe.write_text(FAKE); exe.chmod(0o755); repo.mkdir(); cfg.mkdir()
    def git(*a): return subprocess.run(['git', '-C', str(repo), *a], check=True, capture_output=True, text=True).stdout.strip()
    git('init', '-q'); (repo/'big.py').write_text('x = 1\n'); git('add', '.')
    git('-c', 'user.name=t', '-c', 'user.email=t@example.com', 'commit', '-qm', 'one')
    ls = dict(tool='ls', ok=True, args={'path': str(repo)})
    past = dict(session='s1', model='m1', at=time.time() + 60, reply='big.py 50',
                prompt='<user-request>\nWhat are the sizes of the big files in this repository?\n</user-request>',
                activity=[ls, dict(tool='run_shell', ok=False, args={'command': 'stat big.py'}), ls])
    (cfg/'agent-history.jsonl').write_text(json.dumps(past) + '\n')
    fire = AttrDict(budget=5, cfg=str(cfg), work=str(d/'nights'), exe=str(exe))
    with learned_lock(cfg/'learned'): note = nightly(fire)   # a beat that reclaimed a night still running
    assert note.startswith('a batch started earlier is still running'), note
    s = dict(session='s9', prompt='p', model='m', start=dict(repo=str(repo), head='HEAD', dirty=None), route=(0, 1, 3))
    test_eq(improve_batch(d/'b1', sessions=[s, s | dict(session='s8', model=None)], models='m1,m2', exe=str(exe)), [])   # a list as text: `m` is no model in it

In [ ]:
with TemporaryDirectory() as d:
    d = Path(d)
    exe, repo = d/'fake-ramabana', d/'repo'
    exe.write_text(FAKE); exe.chmod(0o755); repo.mkdir()
    def git(*a): return subprocess.run(['git', '-C', str(repo), *a], check=True, capture_output=True, text=True).stdout.strip()
    git('init', '-q'); (repo/'big.py').write_text('x = 1\n'); git('add', '.')
    git('-c', 'user.name=t', '-c', 'user.email=t@example.com', 'commit', '-qm', 'one')
    start = dict(repo=str(repo), head=git('rev-parse', 'HEAD'), dirty=None)
    def session(sid, model='m1', start=start): return dict(session=sid, prompt=f'sizes of the big files ({sid})', model=model, start=start, route=(0, 1, 3))
    real = dict(prompt='sizes of the big files', reply='big.py 50', activity=[dict(tool='run_shell', ok=True, args={'command': 'stat big.py'}, detail='50')])
    sessions = [session('s1') | dict(turn=real), session('s2'), session('s3', model='local'), session('s4', start=start | dict(repo=str(d/'gone')))]
    rows = improve_batch(d/'batch', learned=d/'learned', sessions=sessions, models=['m1'], exe=str(exe))
    test_eq([(r['session'], r.get('adopted')) for r in rows], [('s1', True), ('s2', True), ('s4', None)])   # s3's model is not offered
    day, first, second = f'dhrona-{datetime.now():%Y-%m-%d}.md', rows[0]['pending'], rows[1]['pending']
    test_eq(sorted(_files(d/'learned')), [])   # nothing reaches HEAD until a person accepts a branch
    test_eq([_git(d/'learned', 'show', f'{b}:{day}').count('\n## ') for b in (first, second)], [1, 2])   # the second branch holds both lessons of the day
    test_eq([_git(d/'learned', 'rev-parse', f'{b}^') for b in (first, second)], [_git(d/'learned', 'rev-parse', 'HEAD')] * 2)   # each is one commit off HEAD
    assert rows[-1]['error'].startswith('RuntimeError: could not clone') and 'does not exist' in rows[-1]['error']
    assert (d/'batch'/'s1'/'evidence'/'original.md').exists() and not (d/'batch'/'s2'/'evidence'/'original.md').exists()   # each round gets its own session
    test_eq(json.loads((d/'batch'/'summary.json').read_text()), json.loads(json.dumps(rows, default=str)))
    test_eq(improve_batch(d/'none', learned=d/'learned', sessions=sessions, models=['absent'], exe=str(exe)), [])
    test_eq(json.loads((d/'none'/'summary.json').read_text()), [])
    test_fail(lambda: improve_batch(d/'bad', learned=d/'learned', sessions=sessions, approve='auto', exe=str(exe)), contains='approve')
    paid = d/'paid-ramabana'
    paid.write_text(FAKE.replace("usage={'cost': 0}", "usage={'cost': 1}")); paid.chmod(0o755)
    rows = improve_batch(d/'capped', learned=d/'learned-capped', sessions=sessions, budget=5, exe=str(paid))
    test_eq([(r['session'], r['cost']) for r in rows], [('s1', 5)])   # two bases, the editor, the candidate and the judge; then the cap stops the batch
    paid.write_text(FAKE.replace("usage={'cost': 0}", "usage={'cost': 1}").replace("'YES\\nIt names the same file and size.'", "''"))
    rows = improve_batch(d/'mute-judge', learned=d/'learned-mute', sessions=sessions[:1], budget=5, exe=str(paid))
    test_eq((rows[0]['cost'], 'judge gave no answer' in rows[0]['error']), (5, True))   # a judge that gave no answer was still paid for
    with learned_lock(d/'learned'):
        test_fail(lambda: improve_batch(d/'locked', learned=d/'learned', sessions=sessions, exe=str(exe)), contains='another batch')
    (d/'learned'/'.kosha').mkdir(); (d/'learned'/'.kosha'/'code.db').write_bytes(b'\x95\x00')
    test_eq(improve_batch(d/'indexed', learned=d/'learned', sessions=[], exe=str(exe)), [])   # Ramabana's untracked index is not an edit
    assert not any((d/f).exists() for f in ('bad', 'locked'))   # refused before the batch had a folder

In [ ]:
with TemporaryDirectory() as d:
    d = Path(d)
    exe, repo, cfg, log = d/'leaky-ramabana', d/'repo', d/'cfg', d/'prompts.log'
    PII = ('ann.lee@example.com', '+1 415 555 0134', 'sk-proj-AbCdEf1234567890AbCdEf1234567890xyz')
    leaky = (FAKE.replace("Path(roots[0], 'AGENTS.md').write_text('Read file sizes from ls.\\n')",
                          "Path(roots[0], 'AGENTS.md').write_text('Read file sizes from ls. Ask ann.lee@example.com.\\n')")
                 .replace("acts, reply = [ls], 'big.py 50'", "acts, reply = [ls], 'big.py 50; call +1 415 555 0134'")
                 .replace("'base-1 call 1: wrote AGENTS.md'", "'base-1 call 1: wrote AGENTS.md, ask ann.lee@example.com'")
                 .replace("import json, sys\n", f"import json, sys\nopen({str(log)!r}, 'a').write(sys.argv[-1] + '\\n=====\\n')\n"))
    exe.write_text(leaky); exe.chmod(0o755); repo.mkdir(); cfg.mkdir()
    def git(*a): return subprocess.run(['git', '-C', str(repo), *a], check=True, capture_output=True, text=True).stdout.strip()
    git('init', '-q'); (repo/'big.py').write_text('x = 1\n'); git('add', '.')
    git('-c', 'user.name=t', '-c', 'user.email=t@example.com', 'commit', '-qm', 'one')
    rem = dict(tool='remember', ok=True, args=dict(text='ls -l gives sizes', cue='file sizes', component='tool', evidence=f'key {PII[2]}'))
    (cfg/'agent-history.jsonl').write_text(json.dumps(dict(session='old', at=time.time(), activity=[rem])) + '\n')
    real = dict(prompt='sizes of the big files', reply=f'big.py 50, sent to {PII[0]}',
                activity=[dict(tool='run_shell', ok=True, args={'command': f'curl -H "Authorization: Bearer {PII[2]}" x'}, detail='50')])
    s = dict(session='s1', prompt='sizes of the big files', model='m1', start=dict(repo=str(repo), head=git('rev-parse', 'HEAD'), dirty=None),
             route=(0, 1, 3), turn=real)
    rows = improve_batch(d/'batch', learned=d/'learned', sessions=[s], exe=str(exe), cfg=cfg)
    assert rows[0]['adopted'], rows
    ev = d/'batch'/'s1'/'evidence'
    assert 'Tool defects' in (ev/'lessons.md').read_text() and 'ls -l gives sizes' in (ev/'lessons.md').read_text()   # the editor reads the lesson
    seen = log.read_text().split('=====')
    told = [t for t in seen if t.strip().startswith(('You improve', 'A person asked'))]   # what the editor and the judge were sent
    branch, name = rows[0]['pending'], f'dhrona-{datetime.now():%Y-%m-%d}.md'
    kept = _git(d/'learned', 'show', branch) + _git(d/'learned', 'show', f'{branch}:{name}')
    for x in PII:
        assert not any(x in t for t in told), x
        assert not any(x in f.read_text() for f in ev.iterdir()), x
        assert x not in kept, x
    assert '[EMAIL_1]' in _git(d/'learned', 'show', f'{branch}:{name}')   # the editor's leak was scrubbed before the lesson was kept

In [ ]:
SHORTEN = r'''#!/usr/bin/env python3
"A stand-in for `ramabana` as the compaction editor: it keeps each `[kept N]` line and writes one short lesson."
import json, re, sys
from pathlib import Path
a = sys.argv[1:]
cfg, prompt = Path(a[a.index('--cfg') + 1]), a[-1]
file = prompt.split('<file>\n')[1].split('\n</file>')[0]
reply = '\n\n'.join(re.findall(r'^\[kept \d+\]$', file, re.M) + ['Read file sizes with ls -l.'])
cfg.mkdir(parents=True, exist_ok=True)
(cfg/'agent-history.jsonl').write_text(json.dumps(dict(prompt=prompt, activity=[], reply=reply, usage={'cost': 0})) + '\n')
'''
with TemporaryDirectory() as d:
    d = Path(d); exe, cfg = d/'shorten-ramabana', d/'cfg'
    exe.write_text(SHORTEN); exe.chmod(0o755)
    learned = cfg/'learned'; learned.mkdir(parents=True); (cfg/'agent-history.jsonl').write_text('')
    (learned/'AGENTS.md').write_text('A rule a person wrote.\n'); learned_repo(learned)
    (learned/'AGENTS.md').write_text('A rule a person wrote.\n\n' + 'Read file sizes with ls -l, not stat. ' * 45 + '\n')
    _git(learned, 'add', '-A'); _git(learned, *_AS_DHRONA, 'commit', '-qm', f'{LESSON}: sizes')
    fire = AttrDict(budget=5, cfg=str(cfg), work=str(d/'nights'), exe=str(exe))
    test_eq(nightly(fire | dict(compact_at=0.9)), 'no session left that an earlier night has not tried')   # 1,735 characters: past 80% of LEARNED_MAX, under 90%
    note = nightly(fire)
    assert 'compaction pending pending/' in note and note.endswith(REVIEW), note
    branch = note.split('compaction pending ')[1].split('\n')[0]
    assert _git(learned, 'show', f'{branch}:AGENTS.md').startswith('A rule a person wrote.\n\nRead file sizes with ls -l.')
    test_eq(nightly(fire), 'no session left that an earlier night has not tried')   # one compaction pending is enough
    _git(learned, 'branch', '-D', branch)
    test_eq(nightly(fire), 'no session left that an earlier night has not tried')   # rejected: not proposed again until HEAD moves
    (learned.parent/'.learned.compacted').unlink()
    rows = improve_batch(d/'batch', learned, sessions=[], exe=str(exe), compact_at=0.5)
    test_eq((rows, json.loads((d/'batch'/'compaction.json').read_text())['ok']), ([], True))
    _git(learned, 'branch', '-D', json.loads((d/'batch'/'compaction.json').read_text())['pending'])
    (learned.parent/'.learned.compacted').unlink()
    exe.write_text(SHORTEN.replace("usage={'cost': 0}", "usage={'cost': 1}"))
    s = dict(session='s1', prompt='p', model='m', start=dict(repo=str(d), head='HEAD', dirty=None), route=(0, 1, 3))
    rows = improve_batch(d/'paid', learned, sessions=[s], exe=str(exe), compact_at=0.5, budget=1)
    test_eq((rows, json.loads((d/'paid'/'compaction.json').read_text())['cost']), ([], 1))   # the compaction spent the budget, so no round started

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()